# # Quant-XAI — One-Shot Revision Re-Run
# **Scientific Reports submission `eb9a1278-f10a-473a-81a8-8a3524df27df`**
#
# Single notebook that regenerates **every** number required by the Editor and Reviewers 1-3,
# plus four self-audit defects found in the submitted artifacts.
#
# ---
# ## Before you press Run
#
# 1. **Settings -> Accelerator -> GPU** (A100 / L4 / P100 / T4x2 all work).
# 2. **Settings -> Internet -> ON**  (required: `timm` downloads pretrained weights; pip installs `onnxruntime`).
# 3. **Add Data** — attach whichever of these you have:
#    - `paddy-disease-classification`  (competition; PRIMARY dataset, Tables 3-8)
#    - `diegopgonzlez/rocoleoriginal`  (RoCoLe, secondary)
#    - `jesperdramsch/siim-acr-pneumothorax-segmentation-data`        (**REQUIRED for SIIM** — the official competition page no longer
#      ships training images, only `stage_2_images/` = test set. Attach the mirror or the
#      ground-truth-localisation cell will skip.)
# 4. **Run `PROFILE = "smoke"` first** (cell 1). ~10 min. It executes every code path end to end on a
#    tiny subset. If smoke is green, `PROFILE = "full"` cannot hit a new code error — only OOM/time.
#
# ## What this fixes (self-audit, not raised by reviewers)
#
# | ID | Defect | Cell |
# |---|---|---|
# | **F-01** | Saliency maps compared at different spatial grids across methods (Grad-CAM 1024 cells vs IG 50176). All metrics now on one 224x224 grid. | 8 |
# | **F-02** | Two MobileNetV3 post-QAT rows are all-ones masks (IoU=Dice=1.0) inflating "+53.9%" to a true ~+0.5%. Collapse rule now excludes them. | 8 |
# | **F-03** | timm `mobilenetv3_large_100` applies `global_pool` BEFORE `conv_head`, so `conv_head` is **1280x1x1**. An auto-picked "last 4-D map" gives a uniform CAM -> fake collapse. Now asserted + swept. | 5, 11 |
# | **F-04** | Calibration drawn from `val_loader` (leakage). Now a seeded stratified TRAIN subset. | 3 |
#
# ## Reviewer coverage map
#
# | Cell | Output | Closes |
# |---|---|---|
# | 1-3 | Config, data, splits, calibration subset | R2#3, F-04 |
# | 4 | FP32 training (Table 3) | baseline |
# | 5 | CAM target-layer resolution (Table N2) | R2#1, R3#6, F-03 |
# | 6 | Fake-quant engine, `legacy` vs `qdq` modes | R3#1, R1#5 |
# | 7 | ONNX INT8 + full-val equivalence + CKA (Table N4) | R2#7, R3#1 |
# | 8 | Full-val drift + k-sweep + random baselines + collapse rate (Tables 6,7,N5,N6,N7) | R1#3, R1#7, R2#2, R3#2, R3#4, F-01, F-02 |
# | 9 | Layer-wise collapse diagnostics (Table N8) | R2#4, R1#4a |
# | 10 | Mixed-precision ablation (Table N9) | R1#4b, R3#6 |
# | 11 | CAM target-layer sweep | R3#6, F-03 |
# | 12 | Calibration ablation MinMax/Entropy/Percentile (Table N3) | R1#5, R2#3 |
# | 13 | QAT lambda sweep + seeds + post-QAT accuracy (Tables 8,N10,N12) | R1#5, R2#6, R3#7 |
# | 14 | Faithfulness: deletion/insertion/conf-drop + randomisation sanity (Table N13) | R3#5 |
# | 15 | SIIM-ACR ground-truth localisation | R3#5 |
# | 16 | Params / FLOPs / size / latency (Table N1) | R1#4c |
# | 17 | Bootstrap CI, paired Wilcoxon, Holm, power (Table N14) | R1#7, R3#2 |
# | 18 | Class-imbalance vs drift correlation | R1#6 |
# | 19 | Figures with software+version stamps | Editor E4 |
# | 20 | Export CSV + LaTeX + requirements.txt + MANIFEST | Editor E5, E6, R1#2 |

In [1]:
# ============================================================================
# CELL 0 - ENVIRONMENT PROBE  (no hard version pins; capability flags instead)
# ============================================================================
import os, sys, gc, json, math, time, random, warnings, subprocess, glob, shutil, platform, itertools
from pathlib import Path
from collections import OrderedDict, defaultdict
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# A 4 h run died asking for 50 MiB while 14.51 GiB was 'in use' but too
# fragmented to serve. expandable_segments lets the allocator grow existing
# segments instead of stranding fixed-size blocks. Must precede `import torch`.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def _pip(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", "--disable-pip-version-check", *pkgs]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        return r.returncode == 0
    except Exception as e:
        print(f"  [pip] {pkgs} -> {e}")
        return False


def ensure(mod, pip_name=None):
    """Import `mod`; try to pip-install if missing. Returns module or None (never raises)."""
    try:
        return __import__(mod)
    except Exception:
        pass
    _pip(pip_name or mod)
    try:
        return __import__(mod)
    except Exception as e:
        print(f"  [WARN] optional dependency '{mod}' unavailable -> dependent cells will SKIP ({type(e).__name__})")
        return None


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

timm = ensure("timm")
if timm is None:
    raise SystemExit("timm is required. Enable Internet in notebook settings and re-run.")

import scipy
from scipy import stats as sstats
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import skimage
from skimage.segmentation import slic
try:
    from skimage.segmentation import quickshift
except Exception:
    quickshift = None

# --- optional ---------------------------------------------------------------
onnx = ensure("onnx")
ort = ensure("onnxruntime")
pydicom = ensure("pydicom")
try:
    import seaborn as sns
except Exception:
    sns = None

CAPS = {
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "onnx": onnx is not None,
    "onnxruntime": ort is not None,
    "pydicom": pydicom is not None,
    "quickshift": quickshift is not None,
    "amp_bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
}

VERSIONS = OrderedDict(
    python=platform.python_version(), torch=torch.__version__, timm=timm.__version__,
    numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
    scikit_image=skimage.__version__, matplotlib=matplotlib.__version__,
    onnx=(onnx.__version__ if onnx else "n/a"), onnxruntime=(ort.__version__ if ort else "n/a"),
    pydicom=(pydicom.__version__ if pydicom else "n/a"), pillow=Image.__version__,
    cuda=(torch.version.cuda or "n/a"), gpu=CAPS["gpu"],
)
print("ENVIRONMENT"); print("-" * 62)
for k, v in VERSIONS.items():
    print(f"  {k:<14} {v}")
print("-" * 62)
print("CAPABILITIES:", {k: v for k, v in CAPS.items() if k != "gpu"})

DEV = torch.device("cuda" if CAPS["cuda"] else "cpu")

# ---------------------------------------------------------------------------
# HARD PREFLIGHT (added after a real run died 40 cells deep on a P100):
# a torch wheel only contains kernels for the SM architectures it was built
# for. torch 2.10+cu128 dropped Pascal (sm_60/sm_61), so on a P100 EVERY cuda
# op raises "no kernel image is available for execution on the device".
# Ask the installed torch what it supports instead of assuming.
# ---------------------------------------------------------------------------
if CAPS["cuda"]:
    _cc = torch.cuda.get_device_capability(0)
    CAPS["sm"] = f"sm_{_cc[0]}{_cc[1]}"
    try:
        CAPS["torch_arch_list"] = list(torch.cuda.get_arch_list())
    except Exception:
        CAPS["torch_arch_list"] = []
    _al = [a for a in CAPS["torch_arch_list"] if a.startswith("sm_")]
    print(f"  {'gpu arch':<14} {CAPS['sm']}")
    print(f"  {'torch built for':<14} {' '.join(_al) or 'unknown'}")
    _bad = bool(_al) and CAPS["sm"] not in _al
    if not _bad:                                  # arch list can lie; prove it
        try:
            _p = torch.zeros(64, 64, device="cuda")
            float((_p @ _p).sum().item())
            torch.nn.Conv2d(3, 8, 3).to("cuda")(torch.zeros(1, 3, 32, 32, device="cuda")).sum().item()
        except Exception as _e:
            _bad = True
            print(f"  live cuda probe FAILED: {type(_e).__name__}: {_e}")
    if _bad:
        raise RuntimeError(
            "\n" + "!" * 76 +
            f"\n  WRONG ACCELERATOR - STOP AND CHANGE IT (nothing below will run)\n"
            f"\n  GPU            : {CAPS['gpu']}  ({CAPS['sm']})"
            f"\n  torch          : {torch.__version__}"
            f"\n  has kernels for: {' '.join(_al) or 'unknown'}"
            f"\n\n  This torch build has no kernels for {CAPS['sm']}, so every CUDA"
            f"\n  operation fails with 'no kernel image is available for execution'."
            f"\n\n  FIX: Notebook -> Settings -> Accelerator -> 'GPU T4 x2'  (T4 = sm_75)"
            f"\n       then Run All again. Do NOT use 'GPU P100' (sm_60) or TPU.\n" +
            "!" * 76)
    print("  cuda kernel probe OK")

ENVIRONMENT
--------------------------------------------------------------
  python         3.12.13
  torch          2.10.0+cu128
  timm           1.0.26
  numpy          2.0.2
  pandas         2.3.3
  scipy          1.16.3
  scikit_image   0.25.2
  matplotlib     3.10.0
  onnx           1.22.0
  onnxruntime    1.28.0
  pydicom        3.0.2
  pillow         11.3.0
  cuda           12.8
  gpu            Tesla T4
--------------------------------------------------------------
CAPABILITIES: {'cuda': True, 'onnx': True, 'onnxruntime': True, 'pydicom': True, 'quickshift': True, 'amp_bf16': True}
  gpu arch       sm_75
  torch built for sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120
  cuda kernel probe OK


In [2]:
# ============================================================================
# CELL 1 - CONFIG  (single source of truth; every knob the reviewers asked about)
# ============================================================================
PROFILE = "full"           # <<<<<<  "full" = the real run. Smoke already passed 26/26.


class CFG:
    seed = 42
    seeds = [42, 1337, 2024]                 # R3#7: multi-seed QAT
    img = 224
    common_grid = 224                        # F-01: ALL maps compared on this single grid
    val_frac = 0.20
    archs = ["tf_efficientnetv2_s", "resnet50", "mobilenetv3_large_100"]
    control_arch = "mobilenetv2_100"         # R3#6: depthwise-separable control
    # ---- training (matches the submitted manuscript) ----
    epochs = 12
    lr = 1e-3
    wd = 1e-4
    warmup = 3
    batch = 48
    # ---- quantization ----
    calib_n = 512                            # R2#3 / F-04: from TRAIN, not val
    calib_methods = ["MinMax", "Entropy", "Percentile"]   # R1#5
    fq_modes = ["legacy", "qdq"]             # R3#1: hook-placement ablation
    fold_bn = True
    # ---- XAI ----
    methods = ["gradcam", "gradcampp", "ig", "lime"]
    ig_steps = 50
    lime_samples = 256                       # resolves code default 500 vs paper 256
    lime_kernel_width = 0.25
    lime_n_segments = 50
    # ---- metrics (R3#4) ----
    k_main = 0.15
    k_sweep = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    collapse_min_levels = 2
    collapse_min_std = 1e-6
    # ---- QAT (R3#7) ----
    qat_epochs = 5
    qat_lr = 1e-4
    lambdas = [0.0, 0.1, 0.5, 1.0, 2.0]      # lambda=0 IS the standard-QAT control
    lambda_main = 0.5
    # ---- evaluation sizes ----
    # Power analysis (N14c): n>=80 for 80% power at delta_rho=0.05, n>=140 with
    # Holm. 320 stratified images is 16x the submitted paper's n=20 and >2x the
    # corrected requirement, and fits one 12 h T4 session. None = full val split.
    n_drift = 320
    n_lime = 120                             # LIME costs 256 forward passes PER image
    n_faith = 500
    n_layerdiag = 128
    n_qat_drift = 400                        # QAT sweep drift subset (paired across all lam/seed)
    # ---- runtime ----
    workers = 2
    amp = True
    qat_train_frac = 0.5                     # QAT fine-tunes on a seeded stratified TRAIN subset
    time_budget_h = 10.0                     # after this, non-essential QAT arms are skipped
    out = Path("/kaggle/working/quantxai")


# ---------------------------------------------------------------------------
# PROFILE INTEGRITY (added after a real run printed PROFILE=full but silently
# used smoke-sized settings, because a find-replace of 'smoke'->'full' had
# rewritten the guard below as well as the PROFILE line).
# Snapshot what CFG DECLARES, then prove 'full' leaves it untouched.
# ---------------------------------------------------------------------------
if PROFILE not in ("smoke", "full"):
    raise RuntimeError(
        f"PROFILE must be exactly 'smoke' or 'full', got {PROFILE!r}. "
        f"Edit ONLY the PROFILE line - never find-replace the word 'smoke', "
        f"which also rewrites the override guard.")

_CFG_DECLARED = {k: getattr(CFG, k) for k in vars(CFG)
                 if not k.startswith('__')}

if PROFILE == "smoke":
    CFG.archs = ["mobilenetv3_large_100"]
    CFG.epochs, CFG.qat_epochs = 2, 1
    CFG.batch = 32
    CFG.seeds = [42]
    CFG.lambdas = [0.0, 0.5]
    CFG.calib_methods = ["MinMax"]
    CFG.calib_n = 96
    CFG.n_drift, CFG.n_lime, CFG.n_faith, CFG.n_layerdiag = 24, 8, 16, 16
    CFG.n_qat_drift, CFG.qat_train_frac, CFG.time_budget_h = 8, 0.10, 999.0
    CFG.ig_steps, CFG.lime_samples = 8, 32
    CFG.k_sweep = [0.10, 0.15, 0.20]
    CFG.control_arch = None

if PROFILE == "full":
    _tampered = {k: (v, getattr(CFG, k)) for k, v in _CFG_DECLARED.items()
                 if getattr(CFG, k) != v}
    if _tampered:
        _lines = "\n".join(f"      CFG.{k}: declared {d!r} -> running with {a!r}"
                             for k, (d, a) in sorted(_tampered.items()))
        raise RuntimeError(
            "PROFILE='full' but CFG was downgraded before the run:\n"
            + _lines +
            "\n\n    Almost always caused by find-replacing 'smoke' -> 'full', "
            "which rewrites\n    the `if PROFILE == \"smoke\":` guard too, so the "
            "smoke overrides fire in full mode.\n"
            "    Re-download a clean notebook and change ONLY the PROFILE line.")

# one honest, unmissable statement of what is about to run
_exp = (f"3 archs / 12 ep / n_drift={CFG.n_drift}" if PROFILE == "full"
        else "1 arch / 2 ep / tiny subsets")
print("=" * 78)
print(f"PROFILE = {PROFILE!r}   (expect: {_exp})")
print(f"  archs      : {CFG.archs}")
print(f"  epochs     : {CFG.epochs} fp32 / {CFG.qat_epochs} qat      batch {CFG.batch}")
print(f"  seeds      : {CFG.seeds}       lambdas {CFG.lambdas}")
print(f"  calib_n    : {CFG.calib_n}     methods {CFG.calib_methods}")
print(f"  n_drift    : {CFG.n_drift} (None = full val split)   n_lime {CFG.n_lime}")
print(f"  ig_steps   : {CFG.ig_steps}    lime_samples {CFG.lime_samples}")
print(f"  time_budget: {CFG.time_budget_h} h")
if PROFILE == "smoke":
    print("  >>> SMOKE RUN: numbers are plumbing checks, NOT results.")
print("=" * 78)

CFG.out.mkdir(parents=True, exist_ok=True)
(CFG.out / "ckpt").mkdir(exist_ok=True)
(CFG.out / "tables").mkdir(exist_ok=True)
(CFG.out / "figures").mkdir(exist_ok=True)
(CFG.out / "onnx").mkdir(exist_ok=True)

RESULTS = {}          # name -> DataFrame, dumped in cell 20
TIMING = {}
SKIPPED = []
T_START = time.time()          # notebook wall clock, drives CFG.time_budget_h

from datetime import datetime, timezone

# ---- version-proof shims (NumPy 2.x removed np.trapz; SciPy <1.9 has no .statistic)
_trapz = getattr(np, "trapezoid", None) or np.trapz


def _stat(res):
    """Return the correlation coefficient from any SciPy version."""
    for a in ("statistic", "correlation"):
        v = getattr(res, a, None)
        if v is not None:
            return float(v)
    try:
        return float(res[0])
    except Exception:
        return float("nan")


def _spearman(a, b):
    a = np.asarray(a, float).ravel(); b = np.asarray(b, float).ravel()
    if a.size < 3 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return float("nan")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return _stat(sstats.spearmanr(a, b))


def strat_sample(df, n_total, seed, col="y"):
    """Deterministic stratified subsample without groupby.apply (pandas-version proof)."""
    if n_total is None or n_total >= len(df):
        return df.reset_index(drop=True)
    rng = np.random.RandomState(seed)
    keep = []
    groups = sorted(df[col].unique().tolist())
    per = max(1, int(n_total // max(1, len(groups))))
    for g in groups:
        idx = np.where(df[col].values == g)[0]
        take = min(per, len(idx))
        keep.extend(rng.choice(idx, take, replace=False).tolist())
    keep = sorted(set(keep))
    return df.iloc[keep].reset_index(drop=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True


def save(name, df):
    RESULTS[name] = df
    df.to_csv(CFG.out / "tables" / f"{name}.csv", index=False)
    print(f"  [saved] {name}  {df.shape}")
    return df


class stage:
    """Context manager: times a stage and converts any failure into a logged SKIP."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t = time.time()
        gpu_free()
        if CAPS.get("cuda"):
            torch.cuda.reset_peak_memory_stats()
        print(f"\n=== {self.name} === [+{(self.t - T_START)/60:.1f} min elapsed] | gpu {gpu_mem()} | ram {ram()}")
        return self
    def __exit__(self, et, ev, tb):
        dt = time.time() - self.t
        TIMING[self.name] = round(dt, 1)
        peak = (torch.cuda.max_memory_allocated() / 2 ** 30) if CAPS.get("cuda") else 0.0
        MEMLOG.append(dict(stage=self.name, sec=round(dt, 1),
                           peak_gib=round(peak, 2), ram=ram(),
                           failed=et is not None))
        gc.collect()               # drop reference cycles before we measure
        gpu_free()                 # never hand fragmentation to the next stage
        if et is None:
            print(f"--- {self.name} done in {dt:.1f}s | peak {peak:.2f} GiB | now {gpu_mem()} | ram {ram()}")
        else:
            SKIPPED.append(f"{self.name} :: {et.__name__}: {ev}")
            print(f"!!! {self.name} SKIPPED after {dt:.1f}s -> {et.__name__}: {ev}")
            import traceback; traceback.print_exc()
        # persist off-box before the next stage gets a chance to kill us
        try:
            _hv = any(s in self.name for s in ("CELL 4 ", "CELL 13", "CELL 20"))
            snapshot(self.name, heavy=_hv, force=_hv)
        except Exception:
            pass
        return True      # never abort the notebook


# ---------------------------------------------------------------------------
# MEMORY + RESUME (added after a 4 h run OOMed and lost all of its work)
# ---------------------------------------------------------------------------
MEMLOG = []


def gpu_guard(tag, need_gib=2.5):
    """Run-8 post-mortem: GPU live memory climbed 0.19 -> 13.94 GiB across the
    notebook and every later allocation failed inside a try/except, so 20 of 21
    QAT runs vanished silently. This reclaims first and refuses to start work it
    cannot finish, instead of dying mid-loop."""
    gc.collect()
    if not CAPS.get("cuda"):
        return float("inf")
    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info()[0] / 2 ** 30
    if free >= need_gib:
        print(f"  [mem] {tag}: {free:.2f} GiB free (need {need_gib})")
        return free
    # reclaim: park every cached model on the host, keep only what is in use
    print(f"  [mem] {tag}: only {free:.2f} GiB free -> offloading cached models")
    for _d in (globals().get("FP32") or {}, globals().get("FQ") or {}):
        for _k, _m in list(_d.items()):
            try:
                _m.to("cpu")
            except Exception:
                pass
    gc.collect(); torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info()[0] / 2 ** 30
    print(f"  [mem] {tag}: {free:.2f} GiB free after offload")
    if free < need_gib:
        raise RuntimeError(
            f"{tag}: {free:.2f} GiB free, need {need_gib} GiB. Refusing to start "
            f"a stage that will OOM halfway and lose its work.")
    return free


def ram():
    """Host RAM is what killed run 7. Log it so a kill is never a mystery again."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f"{vm.used / 2 ** 30:.1f}/{vm.total / 2 ** 30:.1f} GiB"
    except Exception:
        return "n/a"
CACHE = CFG.out / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
# Profile tag. Smoke-profile artefacts (24-image drift caches, 1-epoch QAT weights)
# must NEVER be reused by a full run, so every derived artefact is namespaced.
# FP32 checkpoints are the one exception: full deliberately keeps the untagged name
# so a restored 12-epoch checkpoint from a previous session is still picked up.
PTAG = "" if PROFILE == "full" else "_smoke"
_OOM = getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError


def gpu_free():
    gc.collect()
    if CAPS.get("cuda"):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def gpu_mem():
    if not CAPS.get("cuda"):
        return "cpu"
    a = torch.cuda.memory_allocated() / 2 ** 30
    r = torch.cuda.memory_reserved() / 2 ** 30
    return f"{a:.2f} live / {r:.2f} reserved GiB"


# ---------------------------------------------------------------------------
# DURABLE OFF-BOX CHECKPOINTS
#   Kaggle interactive sessions are EPHEMERAL: when the session ends (or the
#   kernel is killed), /kaggle/working is wiped. Writing to disk after every
#   cell is therefore NOT enough - run 7 did exactly that and still lost 102
#   minutes. The only storage that outlives the session is a Kaggle Dataset.
#   After each stage we version a private dataset with everything produced so
#   far. Next session, warm_start() reads it straight back out of the zips.
#   Needs Add-ons -> Secrets: KAGGLE_USERNAME and KAGGLE_KEY (both attached).
#   Without them this degrades to disk-only and says so, loudly.
# ---------------------------------------------------------------------------
SNAP_DIR = Path("/kaggle/working/_snapshot")
SNAP_SLUG = "quantxai-revision-output"
SNAP_EVERY_S = 900          # at most one push per 15 min for cheap stages
_SNAP_LAST = [0.0]
_SNAP_CREDS = [None]
_SNAP_N = [0]


def _snap_creds():
    if _SNAP_CREDS[0] is None:
        try:
            from kaggle_secrets import UserSecretsClient
            _s = UserSecretsClient()
            os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
            _SNAP_CREDS[0] = True
            print("  [snapshot] Kaggle Secrets found -> off-box checkpoints ENABLED")
        except Exception as e:
            _SNAP_CREDS[0] = False
            print(f"  [snapshot] no Kaggle Secrets ({type(e).__name__}) -> DISK ONLY. "
                  f"If this session dies, the run is lost.")
    return _SNAP_CREDS[0]


def snapshot(tag="", heavy=False, force=False):
    """Persist progress off-box. Never raises, never aborts the run."""
    try:
        (CFG.out / "PROGRESS.json").write_text(json.dumps(dict(
            profile=PROFILE, last_stage=tag,
            elapsed_min=round((time.time() - T_START) / 60, 1),
            timing=TIMING, skipped=SKIPPED,
            tables=sorted(RESULTS), memlog=MEMLOG), indent=2, default=str))
    except Exception:
        pass
    if not force and (time.time() - _SNAP_LAST[0]) < SNAP_EVERY_S:
        return
    if not _snap_creds():
        _SNAP_LAST[0] = time.time()
        return
    try:
        t0 = time.time()
        SNAP_DIR.mkdir(parents=True, exist_ok=True)
        subs = ["tables", "tables_latex", "figures", "cache"] + (["ckpt"] if heavy else [])
        for sub in subs:
            src = CFG.out / sub
            if src.exists():
                shutil.copytree(src, SNAP_DIR / sub, dirs_exist_ok=True)
        for f in ("PROGRESS.json", "MANIFEST.json", "TABLE_INDEX.csv"):
            if (CFG.out / f).exists():
                shutil.copy2(CFG.out / f, SNAP_DIR / f)
        _user = os.environ["KAGGLE_USERNAME"]
        (SNAP_DIR / "dataset-metadata.json").write_text(json.dumps(
            {"title": "Quant-XAI revision output",
             "id": _user + "/" + SNAP_SLUG,
             "licenses": [{"name": "CC0-1.0"}]}, indent=2))
        _msg = "snapshot " + str(_SNAP_N[0]) + " after " + str(tag)
        cmd = ["kaggle", "datasets", "version", "-p", str(SNAP_DIR),
               "-m", _msg, "--dir-mode", "zip"]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=2400)
        if r.returncode != 0:      # dataset does not exist yet -> create it
            r = subprocess.run(["kaggle", "datasets", "create", "-p", str(SNAP_DIR),
                                "--dir-mode", "zip"],
                               capture_output=True, text=True, timeout=2400)
        _SNAP_N[0] += 1
        if r.returncode == 0:
            print(f"  [snapshot] pushed in {time.time() - t0:.0f}s -> "
                  f"kaggle.com/datasets/{_user}/{SNAP_SLUG}")
        else:
            _txt = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
            print("  [snapshot] push FAILED: " + (_txt[-1] if _txt else "unknown"))
    except Exception as e:
        print(f"  [snapshot] skipped: {type(e).__name__}: {e}")
    _SNAP_LAST[0] = time.time()


def cached(name, fn):
    """Disk-memoise an expensive DataFrame so a crash costs one block, not the run.
    Delete /kaggle/working/quantxai/cache/<name>.pkl to force recomputation."""
    cf = CACHE / f"{name}.pkl"
    if cf.exists():
        try:
            d = pd.read_pickle(cf)
            print(f"  [resume] {name}: reused {len(d)} cached rows")
            return d
        except Exception as e:
            print(f"  [resume] {name}: cache unreadable ({e}); recomputing")
    d = fn()
    try:
        d.to_pickle(cf)
    except Exception as e:
        print(f"  [resume] could not cache {name}: {e}")
    # MID-CELL durability. CELL 8c runs ~83 min and CELL 13 runs for hours, so
    # waiting for the stage boundary would risk losing everything computed
    # inside them. Throttled: at most one push per SNAP_EVERY_S.
    snapshot("cache:" + name)
    return d


set_seed(CFG.seed)
print(f"PROFILE={PROFILE} | device={DEV} | archs={CFG.archs} | out={CFG.out}")

PROFILE = 'full'   (expect: 3 archs / 12 ep / n_drift=320)
  archs      : ['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100']
  epochs     : 12 fp32 / 5 qat      batch 48
  seeds      : [42, 1337, 2024]       lambdas [0.0, 0.1, 0.5, 1.0, 2.0]
  calib_n    : 512     methods ['MinMax', 'Entropy', 'Percentile']
  n_drift    : 320 (None = full val split)   n_lime 120
  ig_steps   : 50    lime_samples 256
  time_budget: 10.0 h
PROFILE=full | device=cuda | archs=['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100'] | out=/kaggle/working/quantxai


In [3]:
# ============================================================================
# CELL 2 - DATASET DISCOVERY  (auto-detect whatever is attached; never guesses)
# ============================================================================
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".JPEG", ".PNG")
def _kaggle_roots():
    """Kaggle mounts inputs either at /kaggle/input/<slug> or nested under
    /kaggle/input/{competitions,datasets}/[<owner>/]<slug>. Expand the wrapper
    directories so keyword matching sees the REAL dataset folder name."""
    WRAP = {"competitions", "competition", "datasets", "dataset", "input"}
    frontier = sorted(p for p in glob.glob("/kaggle/input/*") if os.path.isdir(p))
    out = list(frontier)
    for depth in range(2):
        nxt = []
        for p in frontier:
            # level 0: only descend through wrapper dirs; level 1: descend through all
            if depth == 0 and os.path.basename(p).lower() not in WRAP:
                continue
            nxt.extend(sorted(q for q in glob.glob(p + "/*") if os.path.isdir(q)))
        out.extend(nxt); frontier = nxt
    seen, uniq = set(), []
    for p in out:
        if p not in seen:
            seen.add(p); uniq.append(p)
    return uniq


INPUT_ROOTS = _kaggle_roots()


WARM_PATTERNS = (("*_fp32_s*.pt", "ckpt"), ("*_qat_l*.pt", "ckpt"),
                 ("drift_*.pkl", "cache"), ("qatdrift_*.pkl", "cache"),
                 ("T3b_training_history.csv", "tables"))


def warm_start():
    """Restore FP32/QAT checkpoints, drift caches and training history from any
    attached dataset so a re-run after a reset skips work that already finished.
    Searches loose files AND the inside of any attached .zip, so it does not matter
    how the previous session's output was re-uploaded."""
    import fnmatch, zipfile
    found = []

    def want(fname):
        for pat, sub in WARM_PATTERNS:
            if fname and fnmatch.fnmatch(fname, pat):
                return sub
        return None

    # ---- loose files, depth-bounded so we never crawl a whole image tree ----
    for root in INPUT_ROOTS:
        base = root.rstrip("/").count(os.sep)
        for dp, dn, fn in os.walk(root):
            if dp.count(os.sep) - base >= 4:
                dn[:] = []
            dn[:] = [d for d in dn if not d.startswith(".")]
            for f in fn:
                sub = want(f)
                if sub is None:
                    continue
                dst = CFG.out / sub / f
                if not dst.exists():
                    try:
                        shutil.copy2(os.path.join(dp, f), dst); found.append(f)
                    except Exception as e:
                        print(f"  [warm start] {f}: {e}")

    # ---- inside any attached .zip (Kaggle usually auto-extracts, but not always) ----
    for root in INPUT_ROOTS:
        for zp in glob.glob(os.path.join(root, "*.zip")) + glob.glob(os.path.join(root, "*", "*.zip")):
            try:
                with zipfile.ZipFile(zp) as z:
                    for mem in z.namelist():
                        f = os.path.basename(mem)
                        sub = want(f)
                        if sub is None:
                            continue
                        dst = CFG.out / sub / f
                        if not dst.exists():
                            with z.open(mem) as s_, open(dst, "wb") as o_:
                                shutil.copyfileobj(s_, o_)
                            found.append(f)
            except Exception as e:
                print(f"  [warm start] {os.path.basename(zp)}: {e}")

    if found:
        print(f"[WARM START] restored {len(found)} file(s) -> those stages resume "
              f"instead of recomputing:")
        for f in sorted(found):
            print(f"     {f}")
    else:
        print("[WARM START] no prior checkpoints in attached inputs (fresh run)")


warm_start()


def hydrate_prior_outputs():
    """Copy tables / latex / figures / onnx already computed in EARLIER sessions
    out of any attached dataset (plain folder or .zip) into CFG.out, so the
    bundle this run exports is the COMPLETE set, not just what runs today.
    N17* is deliberately excluded: the run-9 values came from a paddy-trained
    classifier applied to chest X-rays and are being recomputed in-domain."""
    import shutil, zipfile
    STALE = ("N17_siim_ground_truth", "N17b_siim_summary")
    WANT = (("tables", ".csv"), ("tables_latex", ".tex"),
            ("figures", ".png"), ("figures", ".pdf"), ("onnx", ".onnx"))
    for sub_, _ in WANT:
        (CFG.out / sub_).mkdir(parents=True, exist_ok=True)
    n = 0
    for root in INPUT_ROOTS:
        for dp, _dn, fn in os.walk(root):
            for f in fn:
                p = os.path.join(dp, f)
                if f.lower().endswith(".zip"):
                    try:
                        with zipfile.ZipFile(p) as z:
                            for mem in z.namelist():
                                base = os.path.basename(mem)
                                if not base or any(t in base for t in STALE):
                                    continue
                                for sub_, ext in WANT:
                                    if ("/" + sub_ + "/") in ("/" + mem) and base.lower().endswith(ext):
                                        dst = CFG.out / sub_ / base
                                        if not dst.exists():
                                            dst.write_bytes(z.read(mem)); n += 1
                    except Exception:
                        pass
                    continue
                if any(t in f for t in STALE):
                    continue
                parent = os.path.basename(dp)
                for sub_, ext in WANT:
                    if parent == sub_ and f.lower().endswith(ext):
                        dst = CFG.out / sub_ / f
                        if not dst.exists():
                            try:
                                shutil.copy2(p, dst); n += 1
                            except Exception:
                                pass
    loaded = 0
    for f in sorted((CFG.out / "tables").glob("*.csv")):
        nm = f.stem
        if nm == "TABLE_INDEX" or nm in RESULTS:
            continue
        try:
            RESULTS[nm] = pd.read_csv(f); loaded += 1
        except Exception:
            pass
    print(f"[hydrate] recovered {n} prior artefacts | {loaded} tables loaded into RESULTS")
    if loaded:
        print("[hydrate] " + ", ".join(sorted(RESULTS)[:12]) + (" ..." if loaded > 12 else ""))


hydrate_prior_outputs()
print("Attached inputs:")
for r in INPUT_ROOTS:
    print("   ", r)
if not INPUT_ROOTS:
    print("   (none)")


def _class_folder_index(root, min_classes=2, min_per_class=5):
    """Find <root>/**/<class>/<image> layouts. Returns (base_dir, DataFrame[path,label])."""
    best = None
    for dirpath, dirnames, _ in os.walk(root):
        if len(dirnames) < min_classes:
            continue
        rows, ok = [], 0
        for d in sorted(dirnames):
            sub = os.path.join(dirpath, d)
            files = [f for f in os.listdir(sub) if f.endswith(IMG_EXT)] if os.path.isdir(sub) else []
            if len(files) >= min_per_class:
                ok += 1
                rows += [(os.path.join(sub, f), d) for f in files]
        if ok >= min_classes and rows and (best is None or len(rows) > len(best[1])):
            best = (dirpath, pd.DataFrame(rows, columns=["path", "label"]))
        if len(dirnames) > 200:
            break
    return best


def _csv_index(root):
    """Fallback: a CSV/JSON with an image-name column + a label column."""
    imgs = {}
    for dp, _, fns in os.walk(root):
        for f in fns:
            if f.endswith(IMG_EXT):
                imgs.setdefault(f, os.path.join(dp, f))
                imgs.setdefault(os.path.splitext(f)[0], os.path.join(dp, f))
    if not imgs:
        return None
    for dp, _, fns in os.walk(root):
        for f in fns:
            if not f.lower().endswith((".csv", ".json")):
                continue
            p = os.path.join(dp, f)
            try:
                df = pd.read_csv(p) if f.lower().endswith(".csv") else pd.json_normalize(json.load(open(p)))
            except Exception:
                continue
            if df.empty or df.shape[1] < 2:
                continue
            cols = {c.lower(): c for c in df.columns}
            icol = next((cols[c] for c in cols if any(t in c for t in ("image", "file", "name", "id", "path"))), None)
            lcol = next((cols[c] for c in cols if any(t in c for t in ("label", "class", "category", "disease", "target", "diagnos"))), None)
            if not icol or not lcol or icol == lcol:
                continue
            rows = []
            for iv, lv in zip(df[icol].astype(str), df[lcol].astype(str)):
                q = imgs.get(iv) or imgs.get(os.path.basename(iv)) or imgs.get(os.path.splitext(os.path.basename(iv))[0])
                if q:
                    rows.append((q, lv))
            if len(rows) >= 50 and 2 <= pd.Series([r[1] for r in rows]).nunique() <= 60:
                return (dp, pd.DataFrame(rows, columns=["path", "label"]))
    return None


def discover(keywords, name):
    for root in INPUT_ROOTS:
        if not any(k in root.lower() for k in keywords):
            continue
        got = _class_folder_index(root) or _csv_index(root)
        if got:
            base, df = got
            print(f"[{name}] {root}\n         layout={base}\n         {len(df)} images / {df.label.nunique()} classes")
            return df.sort_values("path").reset_index(drop=True)
        print(f"[{name}] {root} -> found but could not infer a label layout")
    print(f"[{name}] NOT ATTACHED -> its cells will SKIP")
    return None


DATA = {}
DATA["paddy"] = discover(["paddy"], "PADDY")
DATA["rocole"] = discover(["rocole", "coffee"], "ROCOLE")

# last resort: any attached image-folder dataset becomes PRIMARY
if DATA["paddy"] is None and DATA["rocole"] is None:
    for root in INPUT_ROOTS:
        got = _class_folder_index(root) or _csv_index(root)
        if got:
            DATA["paddy"] = got[1].sort_values("path").reset_index(drop=True)
            print(f"[FALLBACK] using {root} as PRIMARY ({len(DATA['paddy'])} imgs)")
            break

PRIMARY_NAME = "paddy" if DATA["paddy"] is not None else "rocole"
PRIMARY = DATA["paddy"] if DATA["paddy"] is not None else DATA["rocole"]
if PRIMARY is None:
    raise SystemExit("No classification dataset found. Attach paddy-disease-classification and/or rocoleoriginal.")

CLASSES = sorted(PRIMARY.label.unique().tolist())
C2I = {c: i for i, c in enumerate(CLASSES)}
PRIMARY["y"] = PRIMARY.label.map(C2I)
NCLS = len(CLASSES)
print(f"\nPRIMARY: {len(PRIMARY)} images, {NCLS} classes")
print(PRIMARY.label.value_counts().sort_index().to_string())

# ---- SIIM-ACR (R3#5 ground truth) ------------------------------------------
# Three attachment layouts are supported. Whichever is present wins; nothing is
# assumed about folder names. Images are matched to annotations BY ID, then the
# overlap is counted -- so a dataset that has images but no usable labels is
# detected here, not three hours later.
#
#   A) DICOM + relative-RLE csv
#        kaggle.com/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data
#        (dicom-images-train/, dicom-images-test/, train-rle.csv)  [VERIFIED LIVE]
#   B) PNG images + PNG masks
#        kaggle.com/datasets/vbookshelf/pneumothorax-chest-xray-images-and-masks
#   C) The official competition attachment ALONE -> unusable for ground truth:
#        stage_2_images/ holds the 3,205 stage-2 TEST images, while
#        stage_2_train.csv annotates the 12,047 TRAIN images. The two ID sets do
#        not intersect, and download_images.py points at the retired Google Cloud
#        Healthcare store. Cell 15 will SKIP with this reason printed.

SIIM = {"mode": None, "rle": None, "pairs": [], "n_pos": 0, "why": "no SIIM dataset attached"}

_siim_roots = [r for r in INPUT_ROOTS
               if any(t in r.lower() for t in ("siim", "pneumothorax", "chest"))]

if _siim_roots:
    _dcm, _png, _csvs, _maskdirs = {}, {}, [], set()
    for root in _siim_roots:
        for dp, dns, fns in os.walk(root):
            low = os.path.basename(dp).lower()
            for f in fns:
                fl = f.lower()
                if fl.endswith(".dcm"):
                    _dcm.setdefault(Path(f).stem, os.path.join(dp, f))
                elif fl.endswith((".png", ".jpg", ".jpeg")):
                    (_png.setdefault(dp, {}))[Path(f).stem] = os.path.join(dp, f)
                    if "mask" in low:
                        _maskdirs.add(dp)
                elif fl.endswith(".csv"):
                    _csvs.append(os.path.join(dp, f))

    # ---------- layout A: DICOM + RLE ----------
    _best_why = -1
    _csvs = [c for c in _csvs if "submission" not in os.path.basename(c).lower()]
    for c in sorted(_csvs, key=lambda p: ("train" not in os.path.basename(p).lower(),
                                          "rle" not in os.path.basename(p).lower())):
        try:
            t = pd.read_csv(c)
        except Exception:
            continue
        t.columns = [str(x).strip() for x in t.columns]
        idc = next((x for x in t.columns if x.lower().replace(" ", "") in ("imageid", "id")), None)
        rlc = next((x for x in t.columns
                    if x.lower().replace(" ", "") in ("encodedpixels", "rle", "encoded_pixels")), None)
        if idc is None or rlc is None or not _dcm:
            continue
        t[idc] = t[idc].astype(str).str.strip()
        posr = t[t[rlc].astype(str).str.strip() != "-1"]
        hit = sorted(set(posr[idc]) & set(_dcm))
        print(f"[SIIM] {os.path.basename(c)}: {t[idc].nunique()} ids, "
              f"{posr[idc].nunique()} with a mask | DICOMs on disk: {len(_dcm)} | usable overlap: {len(hit)}")
        if hit:
            SIIM.update(mode="dicom_rle", rle=c, n_pos=len(hit),
                        pairs=[(i, _dcm[i]) for i in hit], why="")
            SIIM["table"] = posr[[idc, rlc]].rename(columns={idc: "ImageId", rlc: "EncodedPixels"})
            break
        if posr[idc].nunique() > _best_why:
            _best_why = posr[idc].nunique()
            SIIM["why"] = (f"{os.path.basename(c)} annotates {posr[idc].nunique()} images, but NOT ONE of "
                           f"them is among the {len(_dcm)} DICOMs attached. The competition download ships "
                           f"the stage-2 TEST images; the labelled TRAIN images were only ever available "
                           f"through the now-retired Google Cloud Healthcare store.")

    # ---------- layout B: PNG image/mask pairs ----------
    if SIIM["mode"] is None and _maskdirs:
        best = []
        for md in _maskdirs:
            parent = os.path.dirname(md)
            for imd, stems in _png.items():
                if imd in _maskdirs or os.path.dirname(imd) != parent:
                    continue
                common = sorted(set(stems) & set(_png[md]))
                if len(common) > len(best):
                    best = [(k, stems[k], _png[md][k]) for k in common]
        if best:
            SIIM.update(mode="png_mask", pairs=best, n_pos=len(best), why="")

if SIIM["mode"] == "dicom_rle":
    print(f"[SIIM] OK layout=DICOM+RLE  {SIIM['n_pos']} annotated images usable  rle={SIIM['rle']}")
elif SIIM["mode"] == "png_mask":
    print(f"[SIIM] OK layout=PNG+masks  {SIIM['n_pos']} image/mask pairs usable")
else:
    print("[SIIM] UNUSABLE -> Cell 15 will SKIP.")
    print(f"       reason: {SIIM['why']}")
    print("       fix: Add Input -> kaggle.com/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data")
    print("            (1.64 GB, dicom-images-train/ + train-rle.csv; the old seesee mirror is deleted)")

[WARM START] restored 12 file(s) -> those stages resume instead of recomputing:
     T3b_training_history.csv
     drift_mobilenetv3_large_100_legacy_full.pkl
     drift_mobilenetv3_large_100_qdq_full.pkl
     drift_resnet50_legacy_full.pkl
     drift_resnet50_qdq_full.pkl
     drift_tf_efficientnetv2_s_legacy_full.pkl
     drift_tf_efficientnetv2_s_qdq_full.pkl
     mobilenetv3_large_100_fp32_s42.pt
     qatdrift_tf_efficientnetv2_s_l0.0_s42_full.pkl
     resnet50_fp32_s42.pt
     tf_efficientnetv2_s_fp32_s42.pt
     tf_efficientnetv2_s_qat_l0.0_s42_full.pt
[hydrate] recovered 70 prior artefacts | 26 tables loaded into RESULTS
[hydrate] N13_faithfulness, N13b_randomization_sanity, N14a_bootstrap_ci, N14b_paired_tests_holm, N14c_power_analysis, N15_fakequant_config, N16_cam_layer_sweep, N18_imbalance_vs_drift, N19_dataset_composition, N1_efficiency, N2_cam_target_layers, N3_calibration_ablation ...
Attached inputs:
    /kaggle/input/datasets
    /kaggle/input/datasets/diegopgonzlez
   

In [4]:
# ============================================================================
# CELL 3 - SPLITS, DATASET, LOADERS, CALIBRATION SUBSET
# F-04 / R2#3: calibration comes from a seeded stratified TRAIN subset, never val.
# ============================================================================
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

tr_idx, va_idx = train_test_split(
    np.arange(len(PRIMARY)), test_size=CFG.val_frac, random_state=CFG.seed, stratify=PRIMARY.y.values)
TRAIN_DF = PRIMARY.iloc[tr_idx].reset_index(drop=True)
VAL_DF = PRIMARY.iloc[va_idx].reset_index(drop=True)

# calibration = stratified subset of TRAIN only (leak-free), fixed seed
_n_cal = min(CFG.calib_n, len(TRAIN_DF))
_cal_idx, _ = train_test_split(np.arange(len(TRAIN_DF)), train_size=_n_cal,
                               random_state=CFG.seed, stratify=TRAIN_DF.y.values)
CALIB_DF = TRAIN_DF.iloc[_cal_idx].reset_index(drop=True)

DRIFT_DF = VAL_DF.reset_index(drop=True) if CFG.n_drift is None else strat_sample(VAL_DF, CFG.n_drift, CFG.seed)
LIME_DF = strat_sample(DRIFT_DF, CFG.n_lime, CFG.seed)
# The QAT lambda sweep is 21 fine-tuning runs; scoring each on the FULL val split with
# 50-step IG costs ~4.4M forward+backward passes and cannot fit Kaggle's 12 h wall.
# One fixed stratified subset, reused for every (lam, seed), keeps the comparison PAIRED
# and still far exceeds the n>=140 the Holm-corrected power analysis (Table N14c) requires.
QAT_DF = strat_sample(DRIFT_DF, CFG.n_qat_drift, CFG.seed)

print(f"train={len(TRAIN_DF)}  val={len(VAL_DF)}  calib={len(CALIB_DF)} (from TRAIN)  "
      f"drift={len(DRIFT_DF)}  lime={len(LIME_DF)}  qat_drift={len(QAT_DF)}")


class ImgDS(Dataset):
    def __init__(self, df, train=False, size=CFG.img):
        self.p = df.path.tolist(); self.y = df.y.tolist(); self.train = train; self.size = size
    def __len__(self):
        return len(self.p)
    def _load(self, i):
        im = Image.open(self.p[i]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        return np.asarray(im, np.float32) / 255.0
    def __getitem__(self, i):
        a = self._load(i)
        if self.train:
            if random.random() < 0.5:
                a = a[:, ::-1].copy()
            if random.random() < 0.5:
                a = a[::-1, :].copy()
            k = random.choice([0, 1, 2, 3])
            if k:
                a = np.rot90(a, k).copy()
        x = torch.from_numpy(((a - MEAN) / STD).transpose(2, 0, 1).copy())
        return x, self.y[i], i


def loader(df, train=False, bs=None, shuffle=None):
    return DataLoader(ImgDS(df, train), batch_size=bs or CFG.batch,
                      shuffle=(train if shuffle is None else shuffle),
                      num_workers=CFG.workers, pin_memory=CAPS["cuda"], drop_last=False,
                      persistent_workers=False)


def raw_uint8(path, size=CFG.img):
    return np.asarray(Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR), np.uint8)


def to_tensor(path, size=CFG.img):
    a = raw_uint8(path, size).astype(np.float32) / 255.0
    return torch.from_numpy(((a - MEAN) / STD).transpose(2, 0, 1).copy())


CALIB_X = torch.stack([to_tensor(p) for p in CALIB_DF.path.tolist()])
print("calibration tensor:", tuple(CALIB_X.shape))

train=8325  val=2082  calib=512 (from TRAIN)  drift=320  lime=120  qat_drift=320
calibration tensor: (512, 3, 224, 224)


In [5]:
# ============================================================================
# CELL 4 - FP32 TRAINING  (Table 3). Checkpointed: safe to re-run.
# ============================================================================
def build(arch, ncls=NCLS, pretrained=True):
    try:
        return timm.create_model(arch, pretrained=pretrained, num_classes=ncls)
    except Exception as e:
        print(f"  [WARN] pretrained fetch failed for {arch} ({type(e).__name__}); using random init. "
              f"Turn Internet ON for the real run.")
        return timm.create_model(arch, pretrained=False, num_classes=ncls)


@torch.no_grad()
def evaluate(model, dl, want_logits=False):
    model.eval()
    P, Y, L = [], [], []
    for x, y, _ in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast("cuda", enabled=CFG.amp and CAPS["cuda"]):
            o = model(x)
        o = o.float()
        P.append(o.argmax(1).cpu()); Y.append(y)
        if want_logits:
            L.append(o.cpu())
    P = torch.cat(P).numpy(); Y = torch.cat(Y).numpy()
    from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score
    m = dict(acc=accuracy_score(Y, P), macro_f1=f1_score(Y, P, average="macro", zero_division=0),
             bal_acc=balanced_accuracy_score(Y, P))
    return (m, P, Y, (torch.cat(L).numpy() if want_logits else None))


def train_fp32(arch, seed=CFG.seed):
    ck = CFG.out / "ckpt" / f"{arch}_fp32_s{seed}{PTAG}.pt"
    model = build(arch).to(DEV)
    if ck.exists():
        model.load_state_dict(torch.load(ck, map_location=DEV))
        print(f"  [{arch}] loaded checkpoint")
        return model, None
    set_seed(seed)
    dtr, dva = loader(TRAIN_DF, True), loader(VAL_DF)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
    steps = max(1, len(dtr))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG.lr, total_steps=CFG.epochs * steps,
        pct_start=min(0.9, CFG.warmup / max(CFG.epochs, 1)), anneal_strategy="cos")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=CFG.amp and CAPS["cuda"])
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=CFG.amp and CAPS["cuda"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    best, hist = -1.0, []
    for ep in range(CFG.epochs):
        model.train(); tot = 0.0
        for x, y, _ in dtr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=CFG.amp and CAPS["cuda"]):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * x.size(0)
        m, *_ = evaluate(model, dva)
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(dtr.dataset), **m))
        print(f"  [{arch}] ep{ep+1:02d} loss={hist[-1]['train_loss']:.4f} "
              f"f1={m['macro_f1']:.4f} bal={m['bal_acc']:.4f}")
        if m["macro_f1"] > best:
            best = m["macro_f1"]; torch.save(model.state_dict(), ck)
            snapshot(f"ckpt:{arch}", heavy=True)      # throttled; weights are expensive
    model.load_state_dict(torch.load(ck, map_location=DEV))
    return model, pd.DataFrame(hist)


FP32 = {}
rows, hists = [], []
with stage("CELL 4 - FP32 training (Table 3)"):
    for a in CFG.archs:
        mdl, h = train_fp32(a)
        FP32[a] = mdl
        if h is not None:
            h["arch"] = a; hists.append(h)
        m, *_ = evaluate(mdl, loader(VAL_DF))
        rows.append(dict(arch=a, macro_f1=round(m["macro_f1"], 4),
                         bal_acc=round(m["bal_acc"], 4), acc=round(m["acc"], 4),
                         best_epoch=(int(h.macro_f1.idxmax() + 1) if h is not None else -1)))
    # If every arch loaded from a checkpoint, this session produced no training
    # history, so T3b would silently vanish from MANIFEST/TABLE_INDEX/LaTeX and
    # best_epoch would read -1. Recover both from the previous session's CSV.
    _prev_h = CFG.out / "tables" / "T3b_training_history.csv"
    if not hists and _prev_h.exists():
        h_old = pd.read_csv(_prev_h)
        save("T3b_training_history", h_old)
        _be = {a: int(g.loc[g.macro_f1.idxmax(), "epoch"]) for a, g in h_old.groupby("arch")}
        for r in rows:
            if r["best_epoch"] == -1 and r["arch"] in _be:
                r["best_epoch"] = _be[r["arch"]]
        print(f"  [resume] recovered T3b_training_history ({len(h_old)} rows) from previous session")
    save("T3_fp32_performance", pd.DataFrame(rows))
    if hists:
        save("T3b_training_history", pd.concat(hists, ignore_index=True))


=== CELL 4 - FP32 training (Table 3) === [+6.9 min elapsed] | gpu 0.01 live / 0.02 reserved GiB | ram 2.1/31.3 GiB


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

  [tf_efficientnetv2_s] loaded checkpoint


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  [resnet50] loaded checkpoint


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

  [mobilenetv3_large_100] loaded checkpoint
  [saved] T3b_training_history  (36, 6)
  [resume] recovered T3b_training_history (36 rows) from previous session
  [saved] T3_fp32_performance  (3, 5)
--- CELL 4 - FP32 training (Table 3) done in 53.8s | peak 0.97 GiB | now 0.19 live / 0.20 reserved GiB | ram 2.8/31.3 GiB
  [snapshot] no Kaggle Secrets (BackendError) -> DISK ONLY. If this session dies, the run is lost.


In [6]:
# ============================================================================
# CELL 5 - CAM TARGET LAYER RESOLUTION  (Table N2 | R2#1, R3#6, F-03)
#
# F-03: in timm, MobileNetV3 runs global_pool BEFORE conv_head, so `conv_head`
# is 1280x1x1. A "last 4-D feature map" auto-picker selects it and yields a
# perfectly UNIFORM CAM -> all-ones top-k mask -> IoU==k, rho==0, GradCAM==GradCAM++.
# We therefore (a) prefer an explicit path per arch, (b) HARD-ASSERT H,W >= 4.
# ============================================================================
PREFERRED = {
    "resnet50":              ["layer4", "layer4.2"],
    "tf_efficientnetv2_s":   ["bn2", "conv_head", "blocks"],
    "efficientnetv2_s":      ["bn2", "conv_head", "blocks"],
    "mobilenetv3_large_100": ["blocks", "blocks.6"],     # NOT conv_head (1x1!)
    "mobilenetv2_100":       ["blocks", "conv_head"],
}


def layer_shapes(model, arch):
    shapes, hooks = OrderedDict(), []
    def mk(n):
        def fn(m, i, o):
            if torch.is_tensor(o):
                shapes[n] = tuple(o.shape)
        return fn
    for n, m in model.named_modules():
        if n:
            hooks.append(m.register_forward_hook(mk(n)))
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, 3, CFG.img, CFG.img, device=next(model.parameters()).device))
    for h in hooks:
        h.remove()
    return shapes


def resolve_cam_layer(model, arch, min_spatial=4):
    sh = layer_shapes(model, arch)
    naive = None
    for n, s in sh.items():
        if len(s) == 4:
            naive = (n, s)
    for cand in PREFERRED.get(arch, []):
        s = sh.get(cand)
        if s and len(s) == 4 and s[2] >= min_spatial and s[3] >= min_spatial:
            return cand, s, naive
    best = None
    for n, s in sh.items():
        if len(s) == 4 and s[2] >= min_spatial and s[3] >= min_spatial:
            best = (n, s)
    if best is None:
        raise RuntimeError(f"{arch}: no 4-D feature map with spatial >= {min_spatial}")
    return best[0], best[1], naive


CAM_LAYER = {}
with stage("CELL 5 - CAM target layers (Table N2)"):
    rows = []
    for a, m in FP32.items():
        name, shp, naive = resolve_cam_layer(m, a)
        CAM_LAYER[a] = name
        trap = bool(naive and (naive[1][2] < 4 or naive[1][3] < 4))
        rows.append(dict(arch=a, cam_layer=name, shape=str(shp), channels=shp[1],
                         spatial=f"{shp[2]}x{shp[3]}",
                         naive_last4d=(naive[0] if naive else "n/a"),
                         naive_shape=(str(naive[1]) if naive else "n/a"),
                         naive_is_degenerate=trap))
        print(f"  [{a}] CAM -> {name} {shp}   | naive picker would take "
              f"{naive[0] if naive else 'n/a'} {naive[1] if naive else ''} "
              f"{'<<< DEGENERATE 1x1 (F-03 CONFIRMED)' if trap else ''}")
        assert shp[2] >= 4 and shp[3] >= 4, f"{a}: CAM layer spatial too small"
    save("N2_cam_target_layers", pd.DataFrame(rows))

# --- cascade gate: only architectures that really trained AND resolved a CAM layer
ARCHS_OK = [a for a in CFG.archs if a in FP32 and a in CAM_LAYER]
_NO_MODELS = ("no usable FP32 model: CELL 4 (training) or CELL 5 (CAM layer) failed. "
              "Every table below depends on it - fix that error first.")
_NO_FQ = ("no usable fake-quant model: CELL 6 failed. "
          "All drift/QAT tables depend on it - fix that error first.")
print(f"\n[GATE] trained+resolved architectures = {ARCHS_OK or 'NONE'}")
if len(ARCHS_OK) < len(CFG.archs):
    print(f"[GATE] MISSING: {[a for a in CFG.archs if a not in ARCHS_OK]}")


=== CELL 5 - CAM target layers (Table N2) === [+7.8 min elapsed] | gpu 0.19 live / 0.20 reserved GiB | ram 2.8/31.3 GiB
  [tf_efficientnetv2_s] CAM -> bn2 (1, 1280, 7, 7)   | naive picker would take global_pool.pool (1, 1280, 1, 1) <<< DEGENERATE 1x1 (F-03 CONFIRMED)
  [resnet50] CAM -> layer4 (1, 2048, 7, 7)   | naive picker would take global_pool.pool (1, 2048, 1, 1) <<< DEGENERATE 1x1 (F-03 CONFIRMED)
  [mobilenetv3_large_100] CAM -> blocks (1, 960, 7, 7)   | naive picker would take act2 (1, 1280, 1, 1) <<< DEGENERATE 1x1 (F-03 CONFIRMED)
  [saved] N2_cam_target_layers  (3, 8)
--- CELL 5 - CAM target layers (Table N2) done in 1.5s | peak 2.83 GiB | now 0.19 live / 0.20 reserved GiB | ram 2.7/31.3 GiB

[GATE] trained+resolved architectures = ['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100']


In [7]:
# ============================================================================
# CELL 6 - FAKE-QUANT ENGINE  (R3#1, R1#5)
#   mode "legacy" = the submitted code: hooks on Conv2d/Linear/BN/ReLU/SiLU/Hardswish
#                   -> up to 3 sequential quant-dequant ops per Conv-BN-Act block.
#   mode "qdq"    = ONNX-Runtime-faithful: BN folded into Conv, ONE activation
#                   quantiser per fused op, plus the network input.
# Weights: symmetric per-channel int8 via parametrization (differentiable, STE).
# Activations: asymmetric per-tensor uint8 with a MinMax observer.
# ============================================================================
ACT_T = (nn.ReLU, nn.ReLU6, nn.SiLU, nn.Hardswish, nn.GELU, nn.Hardsigmoid, nn.Hardtanh)
LEGACY_T = (nn.Conv2d, nn.Linear, nn.BatchNorm2d) + ACT_T


class ActFQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("mn", torch.tensor(float("inf")))
        self.register_buffer("mx", torch.tensor(float("-inf")))
        self.observing, self.enabled = True, True
    def forward(self, x):
        if not torch.is_tensor(x) or not x.is_floating_point():
            return x
        if self.observing:
            with torch.no_grad():
                xd = x.detach().float()
                self.mn = torch.minimum(self.mn, xd.min())
                self.mx = torch.maximum(self.mx, xd.max())
            return x
        if not self.enabled or not torch.isfinite(self.mn) or not torch.isfinite(self.mx):
            return x
        mn = torch.clamp(self.mn, max=0.0).to(x.dtype)
        mx = torch.clamp(self.mx, min=0.0).to(x.dtype)
        scale = torch.clamp((mx - mn) / 255.0, min=1e-12)
        zp = torch.round(-mn / scale).clamp(0, 255)
        q = torch.clamp(torch.round(x / scale) + zp, 0, 255)
        return x + ((q - zp) * scale - x).detach()


class WeightFQ(nn.Module):
    """Symmetric per-output-channel int8, straight-through estimator."""
    def __init__(self, qmax=127):
        super().__init__(); self.qmax = qmax
    def forward(self, W):
        dims = tuple(range(1, W.dim()))
        s = torch.clamp(W.detach().abs().amax(dim=dims, keepdim=True) / self.qmax, min=1e-12)
        Wq = torch.clamp(torch.round(W / s), -self.qmax, self.qmax) * s
        return W + (Wq - W).detach()


def fold_conv_bn(model):
    """Fuse Conv2d->BatchNorm2d. Only EXACT nn.BatchNorm2d (timm BatchNormAct2d
    subclasses BN but also applies an activation - folding it would drop the act)."""
    import torch.nn.utils.fusion as fusion
    n = 0
    def rec(mod):
        nonlocal n
        for _, ch in mod.named_children():
            rec(ch)
        if isinstance(mod, nn.Sequential):
            names = [k for k, _ in mod.named_children()]
            mods = list(mod.children())
            i = 0
            while i + 1 < len(mods):
                a, b = mods[i], mods[i + 1]
                if isinstance(a, nn.Conv2d) and type(b) is nn.BatchNorm2d:
                    setattr(mod, names[i], fusion.fuse_conv_bn_eval(a.eval(), b.eval()))
                    setattr(mod, names[i + 1], nn.Identity()); n += 1; i += 2
                else:
                    i += 1
        c, b = getattr(mod, "conv", None), getattr(mod, "bn", None)
        if isinstance(c, nn.Conv2d) and type(b) is nn.BatchNorm2d:
            mod.conv = fusion.fuse_conv_bn_eval(c.eval(), b.eval()); mod.bn = nn.Identity(); n += 1
    rec(model)
    return n


def make_fq(arch, base_state, mode="qdq", calib_x=None, quant_weights=True,
            skip_modules=(), fold=None, ncls=None):
    """Returns (fq_model, info). Never mutates the FP32 model."""
    fold = CFG.fold_bn if fold is None else fold
    m = build(arch, ncls=(NCLS if ncls is None else ncls), pretrained=False).to(DEV)
    m.load_state_dict(base_state); m.eval()
    info = {"mode": mode, "folded": 0, "fold_verified": None}
    if mode == "qdq" and fold:
        ref = torch.randn(2, 3, CFG.img, CFG.img, device=DEV)
        with torch.no_grad():
            y0 = m(ref).float().clone()
        import copy
        bak = copy.deepcopy(m.state_dict())
        try:
            info["folded"] = fold_conv_bn(m)
            with torch.no_grad():
                y1 = m(ref).float()
            ok = torch.allclose(y0, y1, atol=2e-2, rtol=2e-2)
            info["fold_verified"] = bool(ok)
            if not ok:
                m = build(arch, ncls=(NCLS if ncls is None else ncls), pretrained=False).to(DEV); m.load_state_dict(bak); m.eval()
                info["folded"] = 0
                print("    [fold] verification failed -> reverted (BN kept unfused)")
        except Exception as e:
            m = build(arch, ncls=(NCLS if ncls is None else ncls), pretrained=False).to(DEV); m.load_state_dict(bak); m.eval()
            info["folded"] = 0; info["fold_verified"] = False
            print(f"    [fold] {type(e).__name__} -> reverted")

    # weight fake-quant
    import torch.nn.utils.parametrize as P
    nw = 0
    if quant_weights:
        for n, mod in m.named_modules():
            if isinstance(mod, (nn.Conv2d, nn.Linear)) and not any(s in n for s in skip_modules):
                P.register_parametrization(mod, "weight", WeightFQ()); nw += 1
    info["weight_quant_layers"] = nw

    # activation fake-quant
    types = LEGACY_T if mode == "legacy" else ACT_T
    fqs, na = {}, 0
    for n, mod in m.named_modules():
        if not n or any(s in n for s in skip_modules):
            continue
        if isinstance(mod, types) or (mode == "qdq" and isinstance(mod, nn.Linear) and "classifier" not in n and "fc" not in n):
            fq = ActFQ().to(DEV); fqs[n] = fq; na += 1
            mod.register_forward_hook(lambda M, I, O, _f=fq: _f(O))
    if mode == "qdq":
        fq_in = ActFQ().to(DEV); fqs["__input__"] = fq_in
        m.register_forward_pre_hook(lambda M, I, _f=fq_in: (_f(I[0]),) + tuple(I[1:]))
        na += 1
    info["act_quant_points"] = na

    # calibrate
    X = CALIB_X if calib_x is None else calib_x
    for f in fqs.values():
        f.observing = True
    with torch.no_grad():
        for i in range(0, len(X), CFG.batch):
            m(X[i:i + CFG.batch].to(DEV))
    for f in fqs.values():
        f.observing = False
    m._fqs = fqs
    return m, info


FQ_INFO = []
FQ = {}
with stage("CELL 6 - fake-quant models (legacy vs qdq)"):
    if not ARCHS_OK:
        raise RuntimeError(_NO_MODELS)
    for a in ARCHS_OK:
        st = {k: v.clone() for k, v in FP32[a].state_dict().items()}
        for mode in CFG.fq_modes:
            mm, info = make_fq(a, st, mode=mode)
            FQ[(a, mode)] = mm
            info["arch"] = a
            FQ_INFO.append(info)
            print(f"  [{a}/{mode}] act_quant_points={info['act_quant_points']} "
                  f"weight_layers={info['weight_quant_layers']} bn_folded={info['folded']} "
                  f"fold_ok={info['fold_verified']}")
    save("N15_fakequant_config", pd.DataFrame(FQ_INFO))

ARCHS_FQ = [a for a in ARCHS_OK if (a, "qdq") in FQ]
print(f"[GATE] fake-quant-ready architectures = {ARCHS_FQ or 'NONE'}")


=== CELL 6 - fake-quant models (legacy vs qdq) === [+7.8 min elapsed] | gpu 0.19 live / 0.20 reserved GiB | ram 2.7/31.3 GiB
  [tf_efficientnetv2_s/legacy] act_quant_points=383 weight_layers=171 bn_folded=0 fold_ok=None
  [tf_efficientnetv2_s/qdq] act_quant_points=103 weight_layers=171 bn_folded=0 fold_ok=True
  [resnet50/legacy] act_quant_points=156 weight_layers=54 bn_folded=0 fold_ok=None
  [resnet50/qdq] act_quant_points=50 weight_layers=54 bn_folded=4 fold_ok=True
  [mobilenetv3_large_100/legacy] act_quant_points=158 weight_layers=64 bn_folded=0 fold_ok=None
  [mobilenetv3_large_100/qdq] act_quant_points=49 weight_layers=64 bn_folded=0 fold_ok=True
  [saved] N15_fakequant_config  (6, 6)
--- CELL 6 - fake-quant models (legacy vs qdq) done in 19.1s | peak 3.33 GiB | now 0.56 live / 0.63 reserved GiB | ram 3.0/31.3 GiB
[GATE] fake-quant-ready architectures = ['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100']


In [8]:
# ============================================================================
# CELL 8a - XAI METHODS, all native (no captum / lime / pytorch-grad-cam deps)
# Every map is returned at CFG.common_grid x CFG.common_grid, min-max normalised.
# THIS IS THE F-01 FIX: one grid for every method and every model.
# ============================================================================
class CamHook:
    def __init__(self, model, layer):
        self.a = self.g = None
        self.h = dict(model.named_modules())[layer].register_forward_hook(self._f)
    def _f(self, m, i, o):
        self.a = o
        if o.requires_grad:
            o.register_hook(self._b)
    def _b(self, g):
        self.g = g
    def close(self):
        try:
            self.h.remove()
        finally:
            # THE LEAK (run 10): removing the handle does NOT release the
            # captured tensors. With create_graph=True the activation is a node
            # in a SECOND-ORDER graph, so holding self.a pins every intermediate
            # of that batch -> ~1.5 GiB retained per QAT run, monotonically.
            self.a = self.g = None


def _norm01(t):
    mn = t.amin(dim=(1, 2), keepdim=True); mx = t.amax(dim=(1, 2), keepdim=True)
    return (t - mn) / (mx - mn + 1e-12)


def to_grid(t, g=None):
    g = g or CFG.common_grid
    if t.dim() == 2:
        t = t[None]
    t = F.interpolate(t[:, None].float(), size=(g, g), mode="bilinear", align_corners=False)[:, 0]
    return _norm01(t).detach().cpu().numpy()


def cam_maps(model, layer, x, tgt, plusplus=False):
    hk = CamHook(model, layer)
    try:
        model.zero_grad(set_to_none=True)
        x = x.to(DEV).requires_grad_(False)
        out = model(x)
        sc = out.gather(1, tgt.view(-1, 1).to(DEV)).sum()
        sc.backward()
        A, G = hk.a, hk.g
        if A is None or G is None:
            raise RuntimeError("CAM hook captured no gradient")
        A, G = A.float(), G.float()
        if not plusplus:
            w = G.mean(dim=(2, 3), keepdim=True)
        else:
            G2 = G * G; G3 = G2 * G
            den = 2.0 * G2 + A.sum(dim=(2, 3), keepdim=True) * G3
            alpha = torch.where(den.abs() > 1e-12, G2 / den, torch.zeros_like(den))
            w = (alpha * F.relu(G)).sum(dim=(2, 3), keepdim=True)
        cam = F.relu((w * A).sum(dim=1))
        return to_grid(cam)
    finally:
        hk.close()
        model.zero_grad(set_to_none=True)


def ig_maps(model, x, tgt, steps=None):
    steps = steps or CFG.ig_steps
    x = x.to(DEV); tgt = tgt.to(DEV)
    base = torch.zeros_like(x)
    diff = x - base
    total = torch.zeros_like(x)
    for i in range(steps):                       # midpoint Riemann rule
        a = (i + 0.5) / steps
        xi = (base + a * diff).detach().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        sc = model(xi).gather(1, tgt.view(-1, 1)).sum()
        g, = torch.autograd.grad(sc, xi)
        total += g.detach()
    attr = (diff * total / steps).abs().sum(1)
    return to_grid(attr)


_SEG_CACHE = {}


def segments_for(path):
    if path in _SEG_CACHE:
        return _SEG_CACHE[path]
    im = raw_uint8(path)
    seg = None
    if CAPS["quickshift"]:
        for kw in ({"kernel_size": 4, "max_dist": 200, "ratio": 0.2},
                   {"kernel_size": 4, "max_dist": 200, "ratio": 0.2, "channel_axis": -1}):
            try:
                seg = quickshift(im, **kw); break
            except Exception:
                continue
    if seg is None:
        try:
            seg = slic(im, n_segments=CFG.lime_n_segments, compactness=10, start_label=0, channel_axis=-1)
        except TypeError:
            seg = slic(im, n_segments=CFG.lime_n_segments, compactness=10, start_label=0)
    seg = seg.astype(np.int64)
    seg -= seg.min()
    _SEG_CACHE[path] = seg
    return seg


@torch.no_grad()
def lime_map(model, path, tgt, n_samples=None, seed=0, bs=64):
    n_samples = n_samples or CFG.lime_samples
    seg = segments_for(path)
    ns = int(seg.max()) + 1
    rng = np.random.RandomState(seed)
    Z = rng.randint(0, 2, size=(n_samples, ns)).astype(np.float32)
    Z[0, :] = 1.0
    x = to_tensor(path)[None].to(DEV)
    fudge = x.mean(dim=(2, 3), keepdim=True)
    y = np.zeros(n_samples, np.float32)
    # no_grad is ESSENTIAL. LIME only reads softmax outputs, but without it every
    # perturbation batch built a full autograd graph through the model. That was
    # the dominant contributor to the 14.5 GiB OOM that killed a 4 h run.
    with torch.no_grad():
        for i in range(0, n_samples, bs):
            zb = Z[i:i + bs]
            mk = torch.from_numpy(zb[:, seg]).to(DEV).unsqueeze(1)     # (b,1,H,W)
            xb = x * mk + fudge * (1 - mk)
            y[i:i + len(zb)] = torch.softmax(model(xb).float(), 1)[:, int(tgt)].cpu().numpy()
            del mk, xb
    ref = Z[0]
    d = 1.0 - (Z @ ref) / (np.linalg.norm(Z, axis=1) * np.linalg.norm(ref) + 1e-12)
    w = np.exp(-(d ** 2) / (CFG.lime_kernel_width ** 2))
    coef = Ridge(alpha=1.0, fit_intercept=True).fit(Z, y, sample_weight=w).coef_
    m = coef[seg].astype(np.float32)
    return to_grid(torch.from_numpy(m))[0]

In [9]:
# ============================================================================
# CELL 8b - METRICS  (R3#4 + F-01 + F-02)
#   * deterministic RANK-based top-k -> |mask| == round(k*N) exactly, ties broken
#     by index, so no tie-inflation (the submitted code used percentile + ">=").
#   * COLLAPSE rule: <2 distinct levels or std<eps -> rho = NaN, row EXCLUDED
#     from means and counted in `collapse_rate` as a first-class metric.
#   * random-overlap baselines: analytic E[IoU]=k/(2-k), E[Dice]=k, plus
#     spatial-permutation and different-image empirical controls.
# ============================================================================
def topk_mask(h, k):
    flat = np.asarray(h, np.float64).ravel()
    n = flat.size
    m = max(1, int(round(k * n)))
    order = np.lexsort((np.arange(n), -flat))     # value desc, then index asc
    mask = np.zeros(n, bool); mask[order[:m]] = True
    return mask.reshape(np.shape(h)), m


def is_collapsed(h):
    a = np.asarray(h, np.float64)
    return (np.unique(a).size < CFG.collapse_min_levels) or (float(a.std()) < CFG.collapse_min_std)


def drift_pair(hf, hq, k=None):
    k = CFG.k_main if k is None else k
    cf, cq = is_collapsed(hf), is_collapsed(hq)
    mf, nf = topk_mask(hf, k); mq, nq = topk_mask(hq, k)
    inter = np.logical_and(mf, mq).sum(); union = np.logical_or(mf, mq).sum()
    iou = float(inter / union) if union else np.nan
    dice = float(2 * inter / (nf + nq)) if (nf + nq) else np.nan
    if cf or cq:
        rho = np.nan
    else:
        r = _spearman(hf, hq)
        rho = float(r) if np.isfinite(r) else np.nan
    return dict(topk_iou=iou, topk_dice=dice, spearman=rho,
                collapsed_fp32=bool(cf), collapsed_int8=bool(cq),
                collapsed=bool(cf or cq),
                n_levels_fp32=int(np.unique(np.asarray(hf)).size),
                n_levels_int8=int(np.unique(np.asarray(hq)).size))


def random_baselines(k, N, trials=200, seed=0):
    rng = np.random.RandomState(seed)
    m = max(1, int(round(k * N)))
    io, di = [], []
    for _ in range(trials):
        a = np.zeros(N, bool); a[rng.choice(N, m, replace=False)] = True
        b = np.zeros(N, bool); b[rng.choice(N, m, replace=False)] = True
        it = np.logical_and(a, b).sum()
        io.append(it / np.logical_or(a, b).sum()); di.append(2 * it / (2 * m))
    return dict(k=k, analytic_iou=k / (2 - k), analytic_dice=k,
                empirical_iou=float(np.mean(io)), empirical_dice=float(np.mean(di)))


with stage("CELL 8b - random-overlap baselines (Table N6)"):
    N = CFG.common_grid ** 2
    save("N6_random_baselines", pd.DataFrame([random_baselines(k, N, 100, CFG.seed) for k in CFG.k_sweep]))


=== CELL 8b - random-overlap baselines (Table N6) === [+8.2 min elapsed] | gpu 0.56 live / 0.63 reserved GiB | ram 3.0/31.3 GiB
  [saved] N6_random_baselines  (6, 5)
--- CELL 8b - random-overlap baselines (Table N6) done in 1.4s | peak 0.56 GiB | now 0.56 live / 0.63 reserved GiB | ram 3.0/31.3 GiB


In [10]:
# ============================================================================
# CELL 8c - FULL-VALIDATION DRIFT + INLINE k-SWEEP
#   Tables 6, 7, N5, N7  |  R1#3, R1#7, R2#2, R3#2, R3#4, F-01, F-02
#   All six k values are computed in the SAME forward/backward pass, so the
#   k-sweep costs ~0 extra GPU time.
# ============================================================================
def measure_drift(arch, fp32_model, fq_model, mode, df, methods=None, tag="base"):
    methods = methods or CFG.methods
    layer = CAM_LAYER[arch]
    rows = []
    paths = df.path.tolist(); ys = df.y.tolist()
    bs = max(1, min(CFG.batch, 16))
    # ---- Grad-CAM / Grad-CAM++ / IG (batched, OOM-adaptive) ----
    # A fixed batch size is what turned a transient memory spike into a dead run.
    # On OOM we now free, halve the batch and retry the SAME images instead of dying.
    s, nb = 0, 0
    while s < len(paths):
        pb, yb = paths[s:s + bs], ys[s:s + bs]
        try:
            x = torch.stack([to_tensor(p) for p in pb]).to(DEV)
            with torch.no_grad():
                pf = fp32_model(x).float().argmax(1)
                pq = fq_model(x).float().argmax(1)
            tgt = pf.clone()                          # explain the FP32 decision on BOTH models
            maps = {}
            if "gradcam" in methods:
                maps["gradcam"] = (cam_maps(fp32_model, layer, x, tgt, False),
                                   cam_maps(fq_model, layer, x, tgt, False))
            if "gradcampp" in methods:
                maps["gradcampp"] = (cam_maps(fp32_model, layer, x, tgt, True),
                                     cam_maps(fq_model, layer, x, tgt, True))
            if "ig" in methods:
                maps["ig"] = (ig_maps(fp32_model, x, tgt), ig_maps(fq_model, x, tgt))
        except _OOM:
            x = maps = None
            gpu_free()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print(f"    [oom] batch -> {bs}, retrying same images")
            continue
        for j, p in enumerate(pb):
            for meth, (Mf, Mq) in maps.items():
                for k in CFG.k_sweep:
                    r = drift_pair(Mf[j], Mq[j], k)
                    rows.append(dict(run=tag, arch=arch, sim=mode, image=os.path.basename(p),
                                     y=yb[j], xai=meth, k=k,
                                     fp32_pred=int(pf[j]), int8_pred=int(pq[j]),
                                     fp32_correct=bool(int(pf[j]) == yb[j]),
                                     pred_match=bool(int(pf[j]) == int(pq[j])), **r))
        del x, maps
        s += len(pb); nb += 1
        if nb % 20 == 0:
            gpu_free()                            # bound fragmentation during long sweeps
    gpu_free()
    # ---- LIME (per-image, subset) ----
    if "lime" in methods:
        sub = df[df.path.isin(set(LIME_DF.path))] if len(df) > len(LIME_DF) else df
        for p, yy in zip(sub.path.tolist(), sub.y.tolist()):
            x = to_tensor(p)[None].to(DEV)
            with torch.no_grad():
                a = int(fp32_model(x).float().argmax(1)); b = int(fq_model(x).float().argmax(1))
            Mf = lime_map(fp32_model, p, a, seed=CFG.seed)
            Mq = lime_map(fq_model, p, a, seed=CFG.seed)
            for k in CFG.k_sweep:
                r = drift_pair(Mf, Mq, k)
                rows.append(dict(run=tag, arch=arch, sim=mode, image=os.path.basename(p),
                                 y=yy, xai="lime", k=k, fp32_pred=a, int8_pred=b,
                                 fp32_correct=bool(a == yy), pred_match=bool(a == b), **r))
    return pd.DataFrame(rows)


DRIFT = []
with stage("CELL 8c - full-val drift + k-sweep (Tables 6,7,N5,N7)"):
    if not ARCHS_FQ:
        raise RuntimeError(_NO_FQ)
    for a in ARCHS_FQ:
        for mode in CFG.fq_modes:
            t0 = time.time()
            # resume unit: if this block already finished in a previous session it is
            # reloaded from disk, so a crash never costs more than one (arch, mode).
            d = cached(f"drift_{a}_{mode}_{PROFILE}",
                       lambda a=a, mode=mode: measure_drift(
                           a, FP32[a], FQ[(a, mode)], mode, DRIFT_DF, tag="base"))
            DRIFT.append(d)
            n_col = int(d[d.k == CFG.k_main].collapsed.sum())
            print(f"  [{a}/{mode}] {len(d)} rows in {time.time()-t0:.0f}s | "
                  f"collapsed@k={CFG.k_main}: {n_col}/{len(d[d.k==CFG.k_main])}")
    DRIFT = pd.concat(DRIFT, ignore_index=True)
    save("RAW_drift_all", DRIFT)

    main = DRIFT[(DRIFT.k == CFG.k_main)]
    valid = main[~main.collapsed]

    # Table 6 rebuilt: model x method, collapsed rows EXCLUDED from means
    t6 = (valid.groupby(["sim", "arch", "xai"])
          .agg(n=("topk_iou", "size"), iou=("topk_iou", "mean"), iou_sd=("topk_iou", "std"),
               dice=("topk_dice", "mean"), dice_sd=("topk_dice", "std"),
               rho=("spearman", "mean"), rho_sd=("spearman", "std")).reset_index())
    cr = (main.groupby(["sim", "arch", "xai"]).collapsed.mean().rename("collapse_rate").reset_index())
    t6 = t6.merge(cr, on=["sim", "arch", "xai"]).round(4)
    save("T6_drift_by_model_method", t6)

    t7 = (valid.groupby(["sim", "xai"])
          .agg(n=("topk_iou", "size"), iou=("topk_iou", "mean"), iou_sd=("topk_iou", "std"),
               dice=("topk_dice", "mean"), rho=("spearman", "mean"), rho_sd=("spearman", "std")).reset_index())
    t7["cv_iou"] = (t7.iou_sd / t7.iou).round(3)
    bl = {k: random_baselines(k, CFG.common_grid ** 2, 50, CFG.seed) for k in [CFG.k_main]}
    t7["iou_over_chance"] = (t7.iou / bl[CFG.k_main]["analytic_iou"]).round(3)
    save("T7_drift_by_method", t7.round(4))

    ks = (DRIFT[~DRIFT.collapsed].groupby(["sim", "arch", "xai", "k"])
          .agg(iou=("topk_iou", "mean"), dice=("topk_dice", "mean"),
               rho=("spearman", "mean"), n=("topk_iou", "size")).reset_index().round(4))
    save("N5_k_sweep", ks)

    # ranking stability across k (R1#3 / R2#2)
    stab = []
    for sim in ks.sim.unique():
        for dim in ["arch", "xai"]:
            ref = None
            for k in sorted(ks.k.unique()):
                o = ks[(ks.sim == sim) & (ks.k == k)].groupby(dim).iou.mean().sort_values(ascending=False)
                if ref is None:
                    ref = o.index.tolist(); tau = 1.0
                else:
                    cur = o.index.tolist()
                    common = [c for c in ref if c in cur]
                    tau = (_stat(sstats.kendalltau([ref.index(c) for c in common],
                                                    [cur.index(c) for c in common]))
                           if len(common) > 1 else np.nan)
                stab.append(dict(sim=sim, dimension=dim, k=k, order="|".join(o.index.tolist()),
                                 kendall_tau_vs_kmin=round(tau, 3) if tau == tau else np.nan))
    save("N5b_ranking_stability_over_k", pd.DataFrame(stab))

    save("N7_collapse_rates", main.groupby(["sim", "arch", "xai"]).agg(
        n=("collapsed", "size"), collapse_rate=("collapsed", "mean"),
        fp32_collapsed=("collapsed_fp32", "mean"), int8_collapsed=("collapsed_int8", "mean"),
        mean_levels_int8=("n_levels_int8", "mean")).reset_index().round(4))

    save("T5_prediction_agreement", main.drop_duplicates(["sim", "arch", "image"])
         .groupby(["sim", "arch"]).agg(n=("pred_match", "size"), agreement=("pred_match", "mean"),
                                       fp32_accuracy=("fp32_correct", "mean")).reset_index().round(4))

# If CELL 8c raised, DRIFT is still a list and every later cell would die on
# `DRIFT.k`. Coerce to an empty typed frame so downstream cells skip cleanly.
if not isinstance(DRIFT, pd.DataFrame):
    DRIFT = pd.DataFrame(columns=["run", "arch", "sim", "image", "y", "xai", "k",
                                  "fp32_pred", "int8_pred", "fp32_correct",
                                  "topk_iou", "topk_dice", "spearman", "collapsed"])
    print("[GATE] DRIFT is empty (CELL 8c failed) - statistics cells will skip.")


=== CELL 8c - full-val drift + k-sweep (Tables 6,7,N5,N7) === [+8.2 min elapsed] | gpu 0.56 live / 0.63 reserved GiB | ram 3.0/31.3 GiB
  [resume] drift_tf_efficientnetv2_s_legacy_full: reused 6480 cached rows
  [tf_efficientnetv2_s/legacy] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [resume] drift_tf_efficientnetv2_s_qdq_full: reused 6480 cached rows
  [tf_efficientnetv2_s/qdq] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [resume] drift_resnet50_legacy_full: reused 6480 cached rows
  [resnet50/legacy] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [resume] drift_resnet50_qdq_full: reused 6480 cached rows
  [resnet50/qdq] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [resume] drift_mobilenetv3_large_100_legacy_full: reused 6480 cached rows
  [mobilenetv3_large_100/legacy] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [resume] drift_mobilenetv3_large_100_qdq_full: reused 6480 cached rows
  [mobilenetv3_large_100/qdq] 6480 rows in 0s | collapsed@k=0.15: 0/1080
  [saved] RAW_drift_all  (388

In [11]:
# ============================================================================
# CELL 13 - QAT: LAMBDA SWEEP + SEEDS + POST-QAT ACCURACY
#   Tables 8, N10, N12  |  R1#5, R2#6, R3#7, F-02
#   * teacher = FROZEN FP32, explicitly detached (fixes the F-02 paired-reference bug)
#   * target class = teacher-predicted (documented, not guessed)
#   * lambda = 0 IS the standard-QAT control R3#7 asked for
#   * post-QAT macro-F1 / balanced acc / agreement recorded (never computed before)
# ============================================================================
def cam_for_loss(model, layer, x, tgt, hook, create_graph=True):
    # create_graph=True is required for the STUDENT (the CAM term must be
    # differentiable w.r.t. student weights -> second-order grad).
    # For the TEACHER it is pure waste: that CAM is a detached target.
    out = model(x)
    sc = out.gather(1, tgt.view(-1, 1)).sum()
    if not hook.a.requires_grad:
        raise RuntimeError(
            f"cam_for_loss: activation at '{layer}' has no grad_fn, so Grad-CAM "
            f"cannot be differentiated. Every parameter feeding this layer has "
            f"requires_grad=False.")
    g, = torch.autograd.grad(sc, hook.a, create_graph=create_graph,
                             retain_graph=True)
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * hook.a).sum(dim=1))
    mn = cam.amin(dim=(1, 2), keepdim=True); mx = cam.amax(dim=(1, 2), keepdim=True)
    return out, (cam - mn) / (mx - mn + 1e-8)


def run_qat(arch, lam, seed):
    ck = CFG.out / "ckpt" / f"{arch}_qat_l{lam}_s{seed}_{PROFILE}.pt"
    st = {k: v.clone() for k, v in FP32[arch].state_dict().items()}
    student, _ = make_fq(arch, st, mode="qdq")
    if ck.exists():
        student.load_state_dict(torch.load(ck, map_location=DEV), strict=False)
        return student
    set_seed(seed)
    teacher = build(arch, pretrained=False).to(DEV)
    teacher.load_state_dict(st); teacher.eval()
    # The teacher is frozen in every sense that matters for R2#6/F-02:
    #   * eval() mode (no BN/dropout updates)
    #   * never handed to an optimiser -> its weights cannot move
    #   * its CAM is .detach()ed before entering the student loss
    # But requires_grad MUST stay True: Grad-CAM is itself a gradient, so a
    # gradless forward pass makes the target CAM impossible to compute.
    # requires_grad_(False) here is what raised
    #   'element 0 of tensors does not require grad and does not have a grad_fn'.
    for p in teacher.parameters():
        p.requires_grad_(True)
    layer = CAM_LAYER[arch]
    ht, hs = CamHook(teacher, layer), CamHook(student, layer)
    _tp = [p for p in student.parameters() if p.requires_grad]
    if not _tp:
        raise RuntimeError(f"QAT student '{arch}' has zero trainable parameters")
    opt = torch.optim.AdamW(_tp, lr=CFG.qat_lr, weight_decay=CFG.wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    _qtr = (TRAIN_DF if CFG.qat_train_frac >= 1.0 else
            strat_sample(TRAIN_DF, int(round(CFG.qat_train_frac * len(TRAIN_DF))), CFG.seed))
    dtr = loader(_qtr, True, bs=max(8, CFG.batch // 2))
    try:
        for ep in range(CFG.qat_epochs):
            student.train(); tl = tc = 0.0
            for x, y, _ in dtr:
                x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
                with torch.no_grad():
                    tgt = teacher(x).argmax(1)
                if lam > 0:
                    _, cam_t = cam_for_loss(teacher, layer, x, tgt, ht,
                                            create_graph=False)
                    cam_t = cam_t.detach()            # <-- explicit detach (F-02)
                    out_s, cam_s = cam_for_loss(student, layer, x, tgt, hs)
                    l_cam = F.mse_loss(cam_s, cam_t)
                else:
                    # lambda=0 IS the standard-QAT control (R3#7): plain CE,
                    # no CAM term, no teacher CAM, no second-order graph.
                    out_s = student(x)
                    l_cam = torch.zeros((), device=DEV)
                l_ce = crit(out_s, y)
                loss = l_ce + lam * l_cam
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
                tl += l_ce.item(); tc += float(l_cam)
            print(f"    [{arch} lam={lam} s={seed}] ep{ep+1} ce={tl/len(dtr):.4f} cam={tc/len(dtr):.5f}")
    finally:
        ht.close(); hs.close()
        # Break the CamHook <-> bound-method reference cycles explicitly; the
        # cyclic collector was never run on the success path, so these survived.
        for _h in (ht, hs):
            _h.a = _h.g = None
        opt.zero_grad(set_to_none=True)
        del teacher, opt, crit, dtr, _tp, st, ht, hs
        gc.collect(); torch.cuda.empty_cache()
    torch.save(student.state_dict(), ck)
    snapshot(f"qat:{arch} lam={lam} s={seed}", heavy=True)   # throttled
    return student


def qat_preflight(arch):
    """One batch through the exact QAT loss path. Costs ~2 s and fails loudly
    instead of letting a broken graph waste the whole lambda x seed sweep."""
    st = {k: v.clone() for k, v in FP32[arch].state_dict().items()}
    student, _ = make_fq(arch, st, mode="qdq")
    teacher = build(arch, pretrained=False).to(DEV)
    teacher.load_state_dict(st); teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(True)
    layer = CAM_LAYER[arch]
    ht, hs = CamHook(teacher, layer), CamHook(student, layer)
    try:
        x, y, _ = next(iter(loader(TRAIN_DF.head(4), False, bs=2)))
        x, y = x.to(DEV), y.to(DEV)
        student.train()
        with torch.no_grad():
            tgt = teacher(x).argmax(1)
        _, cam_t = cam_for_loss(teacher, layer, x, tgt, ht, create_graph=False)
        cam_t = cam_t.detach()
        out_s, cam_s = cam_for_loss(student, layer, x, tgt, hs)
        loss = nn.CrossEntropyLoss()(out_s, y) + 0.5 * F.mse_loss(cam_s, cam_t)
        if not loss.requires_grad:
            raise RuntimeError("QAT loss has no grad_fn")
        loss.backward()
        ng = sum(1 for p in student.parameters()
                 if p.requires_grad and p.grad is not None and torch.isfinite(p.grad).all())
        if ng == 0:
            raise RuntimeError("QAT backward produced no finite gradients")
        print(f"  [preflight {arch}] loss={loss.item():.4f} "
              f"tensors_with_finite_grad={ng} -> QAT graph OK")
    finally:
        ht.close(); hs.close()
        del teacher, student; torch.cuda.empty_cache()


QAT_DRIFT, QAT_PERF, QAT_FAIL = [], [], []

In [12]:
# ============================================================================
# CELL 13-PRE - lambda=0.5 CAM-CONSISTENCY PREFLIGHT, EVERY ARCHITECTURE
# Run 8 called qat_preflight(ARCHS_FQ[0]) only, so a CAM-graph failure on the
# other two architectures was invisible until the sweep had already burned the
# session. This exercises the exact lambda>0 second-order path on all of them
# for ~20 s each, and refuses to continue if none survives.
# ============================================================================
with stage("CELL 13-PRE - QAT lambda=0.5 preflight (all archs)"):
    if not ARCHS_FQ:
        raise RuntimeError(_NO_FQ)
    PRE_OK, PRE_BAD = [], []
    for a in ARCHS_FQ:
        gpu_guard(f"preflight {a}", 2.5)
        try:
            qat_preflight(a)
            PRE_OK.append(a)
        except Exception as e:
            import traceback
            print(f"  [preflight {a}] FAILED {type(e).__name__}: {e}")
            traceback.print_exc()
            PRE_BAD.append(a)
        gc.collect(); torch.cuda.empty_cache()
    print(f"\n  PREFLIGHT ok={PRE_OK}  failed={PRE_BAD}")
    if not PRE_OK:
        raise RuntimeError(
            "the lambda>0 CAM-consistency path is broken on EVERY architecture. "
            "Fix this before spending 2 h on the sweep - T8/N10/N11 depend on it.")
    if PRE_BAD:
        print(f"  WARNING: {PRE_BAD} will produce no mitigated rows.")


=== CELL 13-PRE - QAT lambda=0.5 preflight (all archs) === [+8.2 min elapsed] | gpu 0.56 live / 0.63 reserved GiB | ram 3.0/31.3 GiB
  [mem] preflight tf_efficientnetv2_s: 13.79 GiB free (need 2.5)
  [preflight tf_efficientnetv2_s] loss=0.5018 tensors_with_finite_grad=452 -> QAT graph OK
  [mem] preflight resnet50: 13.76 GiB free (need 2.5)
  [preflight resnet50] loss=0.9625 tensors_with_finite_grad=153 -> QAT graph OK
  [mem] preflight mobilenetv3_large_100: 13.76 GiB free (need 2.5)
  [preflight mobilenetv3_large_100] loss=0.9958 tensors_with_finite_grad=174 -> QAT graph OK

  PREFLIGHT ok=['tf_efficientnetv2_s', 'resnet50', 'mobilenetv3_large_100']  failed=[]
--- CELL 13-PRE - QAT lambda=0.5 preflight (all archs) done in 13.1s | peak 5.79 GiB | now 0.57 live / 0.65 reserved GiB | ram 3.2/31.3 GiB


In [13]:
with stage("CELL 13 - QAT lambda sweep (Tables 8, N10, N12)"):
    dva = loader(VAL_DF)
    if not ARCHS_FQ:
        raise RuntimeError(_NO_FQ)
    for a in ARCHS_FQ:
        # keep exactly one architecture resident; park the rest on the host.
        for _k, _m in list(FP32.items()):
            _m.to(DEV if _k == a else "cpu")
        for _k, _m in list(FQ.items()):
            _m.to(DEV if _k[0] == a else "cpu")
        try:
            gpu_guard(f"CELL 13 {a}", 3.0)
            mf, *_ = evaluate(FP32[a], dva)
            with torch.no_grad():
                pf = np.concatenate([FP32[a](x.to(DEV)).float().argmax(1).cpu().numpy() for x, _, _ in dva])
            mq, pq, Yv, _ = evaluate(FQ[(a, "qdq")], dva)
        except Exception as _ae:
            # Run 10 died exactly here: mobilenetv3 could not be admitted and the
            # raise destroyed 5 completed EffNet runs that were only in memory.
            print(f"  >>> SKIPPING architecture {a}: {type(_ae).__name__}: {_ae}")
            SKIPPED.append(f"CELL 13 {a} :: {type(_ae).__name__}: {_ae}")
            gc.collect(); torch.cuda.empty_cache()
            continue
        QAT_PERF.append(dict(arch=a, stage="FP32", lam=np.nan, seed=CFG.seed, **{k: round(v, 4) for k, v in mf.items()}, agreement=1.0))
        QAT_PERF.append(dict(arch=a, stage="PTQ-INT8", lam=np.nan, seed=CFG.seed, **{k: round(v, 4) for k, v in mq.items()},
                             agreement=round(float((pf == pq).mean()), 4)))
        for lam in CFG.lambdas:
            for seed in (CFG.seeds if lam == CFG.lambda_main else CFG.seeds[:1]):
                _el = (time.time() - T_START) / 3600.0
                if _el > CFG.time_budget_h and lam != CFG.lambda_main:
                    msg = (f"QAT {a} lam={lam} seed={seed} :: time budget "
                           f"({CFG.time_budget_h} h) exceeded at {_el:.2f} h")
                    SKIPPED.append(msg); print(f"  >>> SKIPPED {msg}")
                    continue
                try:
                    gpu_guard(f"qat {a} lam={lam} s={seed}", 2.5)
                    s = run_qat(a, lam, seed)
                    m, p, _, _ = evaluate(s, dva)
                    QAT_PERF.append(dict(arch=a, stage="QAT-INT8", lam=lam, seed=seed,
                                         **{k: round(v, 4) for k, v in m.items()},
                                         agreement=round(float((pf == p).mean()), 4)))
                    d = cached(f"qatdrift_{a}_l{lam}_s{seed}_{PROFILE}",
                               lambda a=a, _s=s, lam=lam, seed=seed: measure_drift(
                                   a, FP32[a], _s, "qdq", QAT_DF,
                                   methods=["gradcam", "gradcampp", "ig"],
                                   tag=f"qat_l{lam}_s{seed}"))
                    d["lam"] = lam; d["seed"] = seed
                    QAT_DRIFT.append(d)
                    # Crash-proofing: write partial QAT results to disk NOW, so a
                    # later OOM can never again throw away completed runs.
                    try:
                        _td = CFG.out / "tables"
                        pd.DataFrame(QAT_PERF).to_csv(_td / "N12_post_qat_performance.csv", index=False)
                        pd.concat(QAT_DRIFT, ignore_index=True).to_csv(_td / "RAW_qat_drift.csv", index=False)
                    except Exception as _pe:
                        print("    [warn] partial QAT save failed:", _pe)
                    dm = d[d.k == CFG.k_main]
                    print(f"  [{a} lam={lam} s={seed}] f1={m['macro_f1']:.4f} "
                          f"IoU={dm[~dm.collapsed].topk_iou.mean():.4f} "
                          f"collapse={dm.collapsed.mean():.3f}")
                    del s, d
                    gc.collect(); torch.cuda.empty_cache()
                    if CAPS["cuda"]:
                        print(f"    [mem] after {a} lam={lam} s={seed}: "
                              f"{torch.cuda.mem_get_info()[0] / 2 ** 30:.2f} GiB free")
                except Exception as e:
                    import traceback
                    print(f"  [{a} lam={lam} s={seed}] FAILED {type(e).__name__}: {e}")
                    traceback.print_exc()
                    QAT_FAIL.append(dict(arch=a, lam=lam, seed=seed,
                                         err=f"{type(e).__name__}: {e}"))
                    gc.collect(); torch.cuda.empty_cache()
    save("N12_post_qat_performance", pd.DataFrame(QAT_PERF))
    if not QAT_DRIFT:
        raise RuntimeError(
            "every QAT run failed -> Tables 8/N10/N11 cannot be built and "
            "R2#6 + R3#7 stay UNANSWERED. Read the per-run FAILED lines above.")
    if True:
        QD = pd.concat(QAT_DRIFT, ignore_index=True)
        save("RAW_qat_drift", QD)
        qm = QD[QD.k == CFG.k_main]
        agg = (qm[~qm.collapsed].groupby(["arch", "xai", "lam"])
               .agg(n=("topk_iou", "size"), iou=("topk_iou", "mean"), dice=("topk_dice", "mean"),
                    rho=("spearman", "mean")).reset_index())
        agg = agg.merge(qm.groupby(["arch", "xai", "lam"]).collapsed.mean()
                        .rename("collapse_rate").reset_index(), on=["arch", "xai", "lam"])
        save("N10_lambda_sweep", agg.round(4))
        # Table 8 rebuilt: base (lambda from CFG.lambda_main) vs PTQ baseline
        b = DRIFT[(DRIFT.sim == "qdq") & (DRIFT.k == CFG.k_main) & (~DRIFT.collapsed)]
        bb = b.groupby(["arch", "xai"]).agg(base_iou=("topk_iou", "mean"), base_dice=("topk_dice", "mean"),
                                            base_rho=("spearman", "mean"), base_n=("topk_iou", "size")).reset_index()
        mm = agg[agg.lam == CFG.lambda_main].rename(
            columns={"iou": "mit_iou", "dice": "mit_dice", "rho": "mit_rho", "n": "mit_n"})
        t8 = bb.merge(mm.drop(columns=["lam"]), on=["arch", "xai"], how="inner")
        for m in ["iou", "dice", "rho"]:
            t8[f"delta_{m}_pct"] = ((t8[f"mit_{m}"] - t8[f"base_{m}"]) / t8[f"base_{m}"].abs().replace(0, np.nan) * 100).round(1)
        if len(t8) == 0:
            print("  [T8 DIAGNOSTIC] baseline (arch,xai) keys :",
                  sorted({tuple(r) for r in bb[["arch", "xai"]].values}))
            print("  [T8 DIAGNOSTIC] mitigated (arch,xai) keys:",
                  sorted({tuple(r) for r in mm[["arch", "xai"]].values}))
            print("  [T8 DIAGNOSTIC] lambdas present :", sorted(agg.lam.unique().tolist()))
            print(f"  [T8 DIAGNOSTIC] lambda_main required: {CFG.lambda_main}")
            print(f"  [T8 DIAGNOSTIC] failed QAT runs: {len(QAT_FAIL)}")
            for _r in QAT_FAIL[:12]:
                print("      ", _r)
            raise RuntimeError(
                f"T8_qat_mitigation would be EMPTY (no QAT drift at "
                f"lambda={CFG.lambda_main}). That is the paper's headline "
                f"mitigation table and R3#7 + R2#6 depend on it. Refusing to "
                f"save a 0-row table and call the run a success.")
        save("T8_qat_mitigation", t8.round(4))
        sd = (QD[(QD.k == CFG.k_main) & (QD.lam == CFG.lambda_main) & (~QD.collapsed)]
              .groupby(["arch", "xai", "seed"]).topk_iou.mean().reset_index()
              .groupby(["arch", "xai"]).topk_iou.agg(["mean", "std", "count"]).reset_index())
        if len(sd) == 0:
            raise RuntimeError(
                f"N11_seed_variability would be EMPTY: no non-collapsed QAT drift "
                f"at lambda={CFG.lambda_main} across seeds {CFG.seeds}. R3#7 asks "
                f"explicitly for multiple seeds.")
        save("N11_seed_variability", sd.round(4))


=== CELL 13 - QAT lambda sweep (Tables 8, N10, N12) === [+8.4 min elapsed] | gpu 0.57 live / 0.65 reserved GiB | ram 3.2/31.3 GiB
  [mem] CELL 13 tf_efficientnetv2_s: 14.08 GiB free (need 3.0)
  [mem] qat tf_efficientnetv2_s lam=0.0 s=42: 14.08 GiB free (need 2.5)
  [resume] qatdrift_tf_efficientnetv2_s_l0.0_s42_full: reused 5760 cached rows
  [tf_efficientnetv2_s lam=0.0 s=42] f1=0.9736 IoU=0.5066 collapse=0.000
    [mem] after tf_efficientnetv2_s lam=0.0 s=42: 14.08 GiB free
  [mem] qat tf_efficientnetv2_s lam=0.1 s=42: 14.08 GiB free (need 2.5)
    [tf_efficientnetv2_s lam=0.1 s=42] ep1 ce=0.5807 cam=0.01210
    [tf_efficientnetv2_s lam=0.1 s=42] ep2 ce=0.5692 cam=0.01442
    [tf_efficientnetv2_s lam=0.1 s=42] ep3 ce=0.5620 cam=0.01451
    [tf_efficientnetv2_s lam=0.1 s=42] ep4 ce=0.5515 cam=0.01437
    [tf_efficientnetv2_s lam=0.1 s=42] ep5 ce=0.5442 cam=0.01424
  [tf_efficientnetv2_s lam=0.1 s=42] f1=0.9756 IoU=0.5221 collapse=0.000
    [mem] after tf_efficientnetv2_s lam=0.1 s=4

In [14]:
# ============================================================================
# CELL 15 - SIIM-ACR GROUND-TRUTH LOCALISATION  (R3#5)
# Answers "similarity != quality" with real pixel-level masks: does INT8 move the
# explanation AWAY from the annotated pneumothorax, or only jitter it?
# Consumes whichever layout Cell 2 found (DICOM+RLE or PNG+masks) and skips with
# a printed reason if neither is usable. Never aborts the notebook.
# ============================================================================
def rle2mask(rle, w, h):
    m = np.zeros(w * h, np.uint8)
    s = str(rle).split()
    if len(s) < 2:
        return m.reshape(h, w).T
    pos, cur = np.asarray(s[0::2], int), np.asarray(s[1::2], int)
    start = 0
    for p, l in zip(pos, cur):
        start += p
        m[start:start + l] = 1
        start += l
    return m.reshape(w, h).T


def _to_u8(a):
    """Any DICOM/PNG pixel array -> uint8 grayscale, window-safe."""
    a = np.asarray(a).astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    lo, hi = float(a.min()), float(a.max())
    a = (a - lo) / (hi - lo) * 255.0 if hi > lo else np.zeros_like(a)
    return a.astype(np.uint8)


def siim_case(item):
    """Return (image_id, uint8 HxW image, bool HxW ground-truth mask) or None."""
    if SIIM["mode"] == "dicom_rle":
        iid, path = item
        px = pydicom.dcmread(path).pixel_array
        h, w = px.shape[:2]
        gt = np.zeros((h, w), np.uint8)
        for r in SIIM["table"].loc[SIIM["table"].ImageId == iid, "EncodedPixels"]:
            gt |= rle2mask(r, w, h)
        return iid, _to_u8(px), gt.astype(bool)
    iid, ip, mp = item
    im = _to_u8(Image.open(ip).convert("L"))
    mk = np.asarray(Image.open(mp).convert("L").resize((im.shape[1], im.shape[0]),
                                                       Image.NEAREST)) > 127
    return iid, im, mk

In [15]:
with stage("CELL 15 - SIIM-ACR ground-truth localisation"):
    if SIIM["mode"] is None:
        raise RuntimeError(f"no usable SIIM ground truth -> {SIIM['why']}")
    if SIIM["mode"] == "dicom_rle" and not CAPS["pydicom"]:
        raise RuntimeError("pydicom unavailable (Internet OFF?) -> cannot read DICOM")

    # ------------------------------------------------------------------
    # IN-DOMAIN FINE-TUNE.  A paddy-trained classifier has no pneumothorax
    # class, so its Grad-CAM on a chest X-ray is driven by whatever leaf
    # filters happen to fire: run 9 measured IoU/chance = 0.164 and
    # pointing-game = 0.000 over 87 images, i.e. worse than random. That is a
    # domain mismatch, not a finding. We fine-tune a binary pneumothorax head
    # on SIIM itself and THEN measure FP32 vs INT8 explanation drift against
    # the radiologist RLE masks, on images held out of the fine-tune.
    # ------------------------------------------------------------------
    if SIIM["mode"] != "dicom_rle":
        raise RuntimeError("in-domain fine-tune needs the DICOM+RLE layout")
    a = ARCHS_FQ[0] if ARCHS_FQ else CFG.archs[0]
    for _k, _m in list(FP32.items()):
        _m.to("cpu")
    for _k, _m in list(FQ.items()):
        _m.to("cpu")
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    gpu_guard(f"CELL 15 finetune {a}", 3.0)

    items = SIIM["pairs"][:min(96, max(32, CFG.n_faith))]
    _eval_ids = set(str(i) for i, _p in items)
    _pos_ids = set(SIIM["table"].ImageId.astype(str))
    # TRUE negatives only: ids the RLE sheet explicitly marks "-1". The DICOM
    # folders also hold ~1.4k UNLABELLED test images; calling those "healthy"
    # would poison the negative class with ~22% mislabelled positives.
    _neg_ids = None
    try:
        _rt = pd.read_csv(SIIM["rle"])
        _rt.columns = [str(x).strip() for x in _rt.columns]
        _ic = next((x for x in _rt.columns
                    if x.lower().replace(" ", "") in ("imageid", "id")), None)
        _rc = next((x for x in _rt.columns
                    if x.lower().replace(" ", "") in ("encodedpixels", "rle", "encoded_pixels")), None)
        if _ic and _rc:
            _rt[_ic] = _rt[_ic].astype(str).str.strip()
            _neg_ids = set(_rt.loc[_rt[_rc].astype(str).str.strip() == "-1", _ic]) - _pos_ids
    except Exception as _e:
        print("  [ft] WARN could not re-read the RLE sheet:", _e)
    if not _neg_ids:
        print("  [ft] WARN no explicit '-1' rows; falling back to any unmasked DICOM")
    _idx = {}
    for _r in INPUT_ROOTS:
        for _dp, _dn, _fn in os.walk(_r):
            for _f in _fn:
                if _f.lower().endswith(".dcm"):
                    _idx.setdefault(Path(_f).stem, os.path.join(_dp, _f))
    _pos = sorted(i for i in _idx if i in _pos_ids and i not in _eval_ids)
    if _neg_ids:
        _neg = sorted(i for i in _idx if i in _neg_ids and i not in _eval_ids)
    else:
        _neg = sorted(i for i in _idx if i not in _pos_ids and i not in _eval_ids)
    print(f"  [ft] DICOMs indexed={len(_idx)} | train pool pos={len(_pos)} neg={len(_neg)} "
          f"| held-out eval={len(items)}")
    if len(_pos) < 50 or len(_neg) < 50:
        raise RuntimeError(f"not enough SIIM cases: pos={len(_pos)} neg={len(_neg)}")

    _rng = np.random.RandomState(CFG.seed)
    _nside = int(min(700, len(_pos), len(_neg)))
    _sel = ([(_pos[i], 1) for i in _rng.permutation(len(_pos))[:_nside]] +
            [(_neg[i], 0) for i in _rng.permutation(len(_neg))[:_nside]])
    _X = np.zeros((len(_sel), CFG.img, CFG.img), np.uint8)
    _Y = np.zeros(len(_sel), np.int64)
    _keep, _bad = 0, 0
    for _iid, _lab in _sel:
        try:
            _im = _to_u8(pydicom.dcmread(_idx[_iid]).pixel_array)
        except Exception:
            _bad += 1
            continue
        _X[_keep] = np.asarray(Image.fromarray(_im).resize((CFG.img, CFG.img), Image.BILINEAR))
        _Y[_keep] = _lab
        _keep += 1
    _X, _Y = _X[:_keep], _Y[:_keep]
    print(f"  [ft] decoded {_keep} images ({int(_Y.sum())} positive) | unreadable {_bad}")
    if _keep < 100:
        raise RuntimeError(f"only {_keep} SIIM images decoded - cannot fine-tune")

    _sm = build(a, ncls=2, pretrained=False).to(DEV)
    _own = _sm.state_dict()
    _xfer = {k: v for k, v in FP32[a].state_dict().items()
             if k in _own and _own[k].shape == v.shape}
    _sm.load_state_dict(_xfer, strict=False)
    print(f"  [ft] transferred {len(_xfer)}/{len(_own)} backbone tensors from {a}")

    _mean = np.asarray(MEAN, np.float32).reshape(1, 1, 1, -1)
    _std = np.asarray(STD, np.float32).reshape(1, 1, 1, -1)

    def _batch(ix):
        _rgb = np.repeat(_X[ix][..., None], 3, axis=-1).astype(np.float32) / 255.0
        _t = ((_rgb - _mean) / _std).transpose(0, 3, 1, 2)
        return torch.from_numpy(np.ascontiguousarray(_t)).float()

    _opt = torch.optim.AdamW(_sm.parameters(), lr=1e-4, weight_decay=CFG.wd)
    _lossf = nn.CrossEntropyLoss()
    _bs, _nep = 24, 3
    for _ep in range(_nep):
        _sm.train()
        _perm = _rng.permutation(_keep)
        _tot, _n, _corr = 0.0, 0, 0
        for _s0 in range(0, _keep, _bs):
            _b = _perm[_s0:_s0 + _bs]
            _xb = _batch(_b).to(DEV)
            _yb = torch.from_numpy(_Y[_b]).to(DEV)
            _opt.zero_grad(set_to_none=True)
            _out = _sm(_xb)
            _l = _lossf(_out, _yb)
            _l.backward()
            _opt.step()
            _tot += float(_l) * len(_b)
            _n += len(_b)
            _corr += int((_out.argmax(1) == _yb).sum())
            del _xb, _yb, _out, _l
        print(f"  [ft] epoch {_ep + 1}/{_nep}  loss={_tot / max(_n, 1):.4f}  "
              f"train_acc={_corr / max(_n, 1):.4f}")
    _sm.eval()
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()

    _cal = _batch(np.arange(min(64, _keep))).to(DEV)
    SIIM_FP32 = _sm
    SIIM_INT8, _fqi = make_fq(a, {k: v.detach().clone() for k, v in _sm.state_dict().items()},
                              mode="qdq", calib_x=_cal, ncls=2)
    SIIM_INT8 = SIIM_INT8.to(DEV).eval()
    SIIM_LAYER = resolve_cam_layer(SIIM_FP32, a)
    del _cal
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    print(f"  [ft] INT8 twin built: {_fqi}")
    print(f"  architecture={a} (binary pneumothorax head, fine-tuned in-domain)")
    rows, used = [], 0
    ERRS, N_EMPTY, N_FRAC = {}, 0, 0
    for it in items:
        try:
            iid, im8, gt = siim_case(it)
        except Exception as e:
            k = f"{type(e).__name__}: {e}"[:160]
            ERRS[k] = ERRS.get(k, 0) + 1
            continue
        if gt.sum() < 64:                       # ignore near-empty annotations
            N_EMPTY += 1
            continue
        gtr = np.asarray(Image.fromarray(gt.astype(np.uint8) * 255)
                         .resize((CFG.common_grid, CFG.common_grid), Image.NEAREST)) > 127
        frac = float(gtr.mean())
        if frac < 0.002 or frac > 0.6:
            N_FRAC += 1
            continue
        rgb = np.stack([np.asarray(Image.fromarray(im8).resize((CFG.img, CFG.img),
                                                               Image.BILINEAR))] * 3, -1).astype(np.float32) / 255.0
        x = torch.from_numpy(((rgb - MEAN) / STD).transpose(2, 0, 1)[None].copy()).float().to(DEV)
        used += 1
        for tag, mdl in (("fp32", SIIM_FP32), ("int8-qdq", SIIM_INT8)):
            with torch.no_grad():
                t = int(mdl(x).float().argmax(1))
            hm = cam_maps(mdl, SIIM_LAYER, x, torch.tensor([t]))[0]
            mk, _ = topk_mask(hm, frac)         # k matched to the lesion size
            inter = int(np.logical_and(mk, gtr).sum())
            union = int(np.logical_or(mk, gtr).sum())
            rows.append(dict(image=str(iid)[-24:], model=tag, gt_frac=round(frac, 4),
                             gt_iou=round(inter / union, 4) if union else np.nan,
                             gt_dice=round(2 * inter / (mk.sum() + gtr.sum()), 4),
                             chance_iou=round(frac / (2 - frac), 4),
                             pointing_game=bool(gtr.ravel()[int(np.argmax(hm))]),
                             mass_in_gt=round(float(hm[gtr].sum() / (hm.sum() + 1e-12)), 4),
                             collapsed=bool(is_collapsed(hm))))
    if ERRS:
        print("  [siim] per-image read errors, most common first:")
        for k, v in sorted(ERRS.items(), key=lambda kv: -kv[1])[:6]:
            print(f"      {v:4d} x {k}")
    print(f"  [siim] tried={len(items)} used={used} read_errors={sum(ERRS.values())} "
          f"empty_mask={N_EMPTY} frac_rejected={N_FRAC}")
    if not rows:
        raise RuntimeError(
            f"no SIIM case survived: tried={len(items)} read_errors={sum(ERRS.values())} "
            f"empty_mask={N_EMPTY} frac_rejected={N_FRAC}. The per-image errors are "
            f"printed above - run 8 discarded them, which is why this took 0.7 s and "
            f"said nothing useful.")
    df = pd.DataFrame(rows)
    save("N17_siim_ground_truth", df)
    summ = (df.groupby("model")[["gt_iou", "gt_dice", "chance_iou", "pointing_game",
                                 "mass_in_gt", "collapsed"]].mean().round(4).reset_index())
    summ["iou_over_chance"] = (summ.gt_iou / summ.chance_iou).round(3)
    summ["n"] = used
    save("N17b_siim_summary", summ)
    print(summ.to_string(index=False))
    print("  NOTE: localisation QUALITY vs radiologist annotation, from a model")
    print("        fine-tuned IN DOMAIN on held-out SIIM images. Answers R3#5")
    print("        'similarity is not quality' and adds a second imaging domain.")
    del SIIM_FP32, SIIM_INT8, _sm, _X, _Y
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    for _k, _m in list(FP32.items()):
        _m.to(DEV)
    for _k, _m in list(FQ.items()):
        _m.to(DEV)
    print("  [ft] paddy models restored to GPU for the figure cell")


=== CELL 15 - SIIM-ACR ground-truth localisation === [+239.9 min elapsed] | gpu 0.08 live / 0.11 reserved GiB | ram 4.1/31.3 GiB
  [mem] CELL 15 finetune tf_efficientnetv2_s: 14.32 GiB free (need 3.0)
  [ft] DICOMs indexed=12089 | train pool pos=2283 neg=8296 | held-out eval=96
  [ft] decoded 1400 images (700 positive) | unreadable 0
  [ft] transferred 780/782 backbone tensors from tf_efficientnetv2_s
  [ft] epoch 1/3  loss=0.6170  train_acc=0.7000
  [ft] epoch 2/3  loss=0.0917  train_acc=0.9821
  [ft] epoch 3/3  loss=0.0163  train_acc=0.9957
  [ft] INT8 twin built: {'mode': 'qdq', 'folded': 0, 'fold_verified': True, 'weight_quant_layers': 171, 'act_quant_points': 103}
  architecture=tf_efficientnetv2_s (binary pneumothorax head, fine-tuned in-domain)
!!! CELL 15 - SIIM-ACR ground-truth localisation SKIPPED after 204.3s -> KeyError: ('bn2', (1, 1280, 7, 7), ('global_pool.pool', (1, 1280, 1, 1)))


Traceback (most recent call last):
  File "/tmp/ipykernel_58/2046651570.py", line 165, in <cell line: 0>
    hm = cam_maps(mdl, SIIM_LAYER, x, torch.tensor([t]))[0]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/3650622699.py", line 41, in cam_maps
    hk = CamHook(model, layer)
         ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/3650622699.py", line 9, in __init__
    self.h = dict(model.named_modules())[layer].register_forward_hook(self._f)
             ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
KeyError: ('bn2', (1, 1280, 7, 7), ('global_pool.pool', (1, 1280, 1, 1)))


In [16]:
# ============================================================================
# CELL 17 - STATISTICS: BOOTSTRAP CI, PAIRED TESTS, HOLM, POWER
#   Table N14  |  R1#2, R1#7, R2#2, R3#3, F-06  (no statsmodels dependency)
# ============================================================================
def boot_ci(v, n=5000, alpha=0.05, seed=0):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if v.size < 2:
        return (np.nan, np.nan, np.nan)
    rng = np.random.RandomState(seed)
    b = v[rng.randint(0, v.size, (n, v.size))].mean(1)
    return float(v.mean()), float(np.percentile(b, 100 * alpha / 2)), float(np.percentile(b, 100 * (1 - alpha / 2)))


def holm(pvals):
    p = np.asarray(pvals, float)
    o = np.argsort(p); n = p.size
    adj = np.empty(n); run = 0.0
    for i, idx in enumerate(o):
        run = max(run, (n - i) * p[idx])
        adj[idx] = min(1.0, run)
    return adj


def cliffs_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return np.nan
    gt = int((a[:, None] > b[None, :]).sum())
    lt = int((a[:, None] < b[None, :]).sum())
    return float((gt - lt) / (a.size * b.size))


def n_for_power(d, power=0.80, alpha=0.05):
    if not np.isfinite(d) or abs(d) < 1e-9:
        return np.inf
    za, zb = sstats.norm.ppf(1 - alpha / 2), sstats.norm.ppf(power)
    return float(np.ceil(((za + zb) / abs(d)) ** 2))


def achieved_power(d, n, alpha=0.05):
    if not np.isfinite(d) or n < 2:
        return np.nan
    za = sstats.norm.ppf(1 - alpha / 2)
    return float(sstats.norm.cdf(abs(d) * np.sqrt(n) - za))


with stage("CELL 17 - statistics (Table N14)"):
    m = DRIFT[(DRIFT.k == CFG.k_main) & (DRIFT.sim == "qdq")]
    v = m[~m.collapsed]
    ci = []
    for (a, x), g in v.groupby(["arch", "xai"]):
        for met in ("topk_iou", "topk_dice", "spearman"):
            mu, lo, hi = boot_ci(g[met].values, seed=CFG.seed)
            ci.append(dict(arch=a, xai=x, metric=met, n=int(g[met].notna().sum()),
                           mean=round(mu, 4), ci_lo=round(lo, 4), ci_hi=round(hi, 4),
                           sig_vs_zero=bool(np.isfinite(lo) and (lo > 0 or hi < 0))))
    save("N14a_bootstrap_ci", pd.DataFrame(ci))

    # paired Wilcoxon across methods on the SAME images (this is the correct test)
    piv = v.pivot_table(index=["arch", "image"], columns="xai", values="topk_iou")
    meths = [c for c in piv.columns]
    tests = []
    for i in range(len(meths)):
        for j in range(i + 1, len(meths)):
            d = piv[[meths[i], meths[j]]].dropna()
            if len(d) < 6:
                continue
            try:
                st, p = sstats.wilcoxon(d[meths[i]], d[meths[j]])
            except Exception:
                st, p = np.nan, np.nan
            diff = (d[meths[i]] - d[meths[j]]).values
            dz = float(diff.mean() / (diff.std(ddof=1) + 1e-12))
            tests.append(dict(comparison=f"{meths[i]} vs {meths[j]}", n_pairs=len(d),
                              mean_diff=round(float(diff.mean()), 4), cohens_dz=round(dz, 3),
                              cliffs_delta=round(cliffs_delta(d[meths[i]], d[meths[j]]), 3),
                              p_raw=p, n_for_80pct_power=n_for_power(dz),
                              achieved_power=round(achieved_power(dz, len(d)), 3)))
    if tests:
        T = pd.DataFrame(tests)
        T["p_holm"] = holm(T.p_raw.fillna(1.0).values)
        T["significant_holm"] = T.p_holm < 0.05
        save("N14b_paired_tests_holm", T.round(5))
        print(T[["comparison", "n_pairs", "mean_diff", "p_raw", "p_holm", "significant_holm"]].to_string(index=False))

    # was the ORIGINAL n=20 design adequate? (F-06)
    pw = []
    for met in ("topk_iou", "spearman"):
        sd = float(v[met].std())
        for delta in (0.02, 0.05, 0.10):
            dz = delta / (sd + 1e-12)
            pw.append(dict(metric=met, observed_sd=round(sd, 4), target_delta=delta,
                           cohens_d=round(dz, 3), n_needed_80=n_for_power(dz),
                           power_at_n20=round(achieved_power(dz, 20), 3),
                           power_at_actual_n=round(achieved_power(dz, int(v[met].notna().sum())), 3),
                           actual_n=int(v[met].notna().sum())))
    save("N14c_power_analysis", pd.DataFrame(pw))

    # QAT effect: paired base vs mitigated on identical images
    if "RAW_qat_drift" in RESULTS:
        Q = RESULTS["RAW_qat_drift"]
        q = Q[(Q.k == CFG.k_main) & (Q.lam == CFG.lambda_main) & (Q.seed == CFG.seed)]
        rows = []
        for (a, x), g in q.groupby(["arch", "xai"]):
            b = m[(m.arch == a) & (m.xai == x)][["image", "topk_iou", "spearman", "collapsed"]]
            j = b.merge(g[["image", "topk_iou", "spearman", "collapsed"]], on="image", suffixes=("_base", "_qat"))
            jj = j[(~j.collapsed_base) & (~j.collapsed_qat)]
            if len(jj) < 6:
                continue
            try:
                _, p = sstats.wilcoxon(jj.topk_iou_base, jj.topk_iou_qat)
            except Exception:
                p = np.nan
            rows.append(dict(arch=a, xai=x, n_paired=len(jj),
                             iou_base=round(jj.topk_iou_base.mean(), 4),
                             iou_qat=round(jj.topk_iou_qat.mean(), 4),
                             delta=round(jj.topk_iou_qat.mean() - jj.topk_iou_base.mean(), 4),
                             collapse_base=round(j.collapsed_base.mean(), 3),
                             collapse_qat=round(j.collapsed_qat.mean(), 3), p_raw=p))
        if rows:
            R = pd.DataFrame(rows); R["p_holm"] = holm(R.p_raw.fillna(1.0).values)
            save("N14d_qat_paired_tests", R.round(5))


=== CELL 17 - statistics (Table N14) === [+243.3 min elapsed] | gpu 0.41 live / 0.53 reserved GiB | ram 4.3/31.3 GiB
  [saved] N14a_bootstrap_ci  (36, 8)
  [saved] N14b_paired_tests_holm  (6, 10)
          comparison  n_pairs  mean_diff         p_raw        p_holm  significant_holm
gradcam vs gradcampp      960    -0.0565  4.182187e-11  1.254656e-10              True
       gradcam vs ig      960     0.2163 8.370447e-124 4.185223e-123              True
     gradcam vs lime      360     0.0312  3.430276e-02  3.430276e-02              True
     gradcampp vs ig      960     0.2728 4.244893e-152 2.546936e-151              True
   gradcampp vs lime      360     0.0891  7.311379e-08  1.462276e-07              True
          ig vs lime      360    -0.1788  2.296574e-29  9.186297e-29              True
  [saved] N14c_power_analysis  (6, 8)
  [saved] N14d_qat_paired_tests  (9, 10)
--- CELL 17 - statistics (Table N14) done in 1.2s | peak 0.41 GiB | now 0.40 live / 0.52 reserved GiB | ram 4.3/31.

In [17]:
# ============================================================================
# CELL 18 - CLASS IMBALANCE vs DRIFT  (R1#6) + per-class drift (Table S2 rebuilt)
# ============================================================================
with stage("CELL 18 - class imbalance vs drift (R1#6)"):
    cnt = PRIMARY.groupby("y").size().rename("n_train").reset_index()
    mm = DRIFT[(DRIFT.k == CFG.k_main) & (DRIFT.sim == "qdq") & (~DRIFT.collapsed)]
    per = mm.groupby(["arch", "y"]).agg(n=("topk_iou", "size"), iou=("topk_iou", "mean"),
                                        rho=("spearman", "mean")).reset_index().merge(cnt, on="y")
    per["class"] = per.y.map(lambda i: CLASSES[i])
    save("S2_per_class_drift", per.round(4))
    cr = []
    for a, g in per.groupby("arch"):
        if len(g) > 2:
            r1 = sstats.spearmanr(g.n_train.values, g.iou.values)
            r2 = sstats.spearmanr(g.n_train.values, g.rho.values)
            cr.append(dict(arch=a, n_classes=len(g),
                           rho_freq_vs_iou=round(_stat(r1), 3), p_iou=round(float(r1.pvalue), 4),
                           rho_freq_vs_spearman=round(_stat(r2), 3), p_rho=round(float(r2.pvalue), 4)))
    save("N18_imbalance_vs_drift", pd.DataFrame(cr))
    print(pd.DataFrame(cr).to_string(index=False) if cr else "  (insufficient classes)")


=== CELL 18 - class imbalance vs drift (R1#6) === [+243.3 min elapsed] | gpu 0.40 live / 0.52 reserved GiB | ram 4.3/31.3 GiB
  [saved] S2_per_class_drift  (30, 7)
  [saved] N18_imbalance_vs_drift  (3, 6)
                 arch  n_classes  rho_freq_vs_iou  p_iou  rho_freq_vs_spearman  p_rho
mobilenetv3_large_100         10           -0.358 0.3104                -0.430 0.2145
             resnet50         10           -0.261 0.4671                -0.491 0.1497
  tf_efficientnetv2_s         10           -0.333 0.3466                -0.261 0.4671
--- CELL 18 - class imbalance vs drift (R1#6) done in 0.3s | peak 0.40 GiB | now 0.40 live / 0.52 reserved GiB | ram 4.3/31.3 GiB


In [18]:
# ============================================================================
# CELL 19 - FIGURES  (Editor E4: every figure regenerated from THIS run's data,
# each stamped with the git-free version fingerprint so figure == table == code)
# ============================================================================
STAMP = f"torch {VERSIONS.get('torch','?')} | timm {VERSIONS.get('timm','?')} | seed {CFG.seed} | k={CFG.k_main} | profile={PROFILE}"


def _fin(fig, name):
    fig.text(0.005, 0.005, STAMP, fontsize=5, color="#666")
    p = CFG.out / "figures" / f"{name}.png"
    fig.savefig(p, dpi=600, bbox_inches="tight")
    try:
        fig.savefig(CFG.out / "figures" / f"{name}.pdf", bbox_inches="tight")
    except Exception as _e:
        print("    [warn] vector copy failed:", _e)
    plt.close(fig)
    print("  figure:", p.name, "(600 dpi png + pdf)")


with stage("CELL 19 - figures"):
    # F1 - drift by method with bootstrap CI (replaces the CI-free bar charts)
    if "N14a_bootstrap_ci" in RESULTS:
        d = RESULTS["N14a_bootstrap_ci"]
        for met in ("topk_iou", "spearman"):
            s = d[d.metric == met]
            if s.empty:
                continue
            g = s.groupby("xai")[["mean", "ci_lo", "ci_hi"]].mean().sort_values("mean", ascending=False)
            fig, ax = plt.subplots(figsize=(5.2, 3.2))
            ax.bar(g.index, g["mean"], color="#4C78A8",
                   yerr=[g["mean"] - g.ci_lo, g.ci_hi - g["mean"]], capsize=4)
            if met == "topk_iou":
                ax.axhline(CFG.k_main / (2 - CFG.k_main), ls="--", c="crimson", lw=1,
                           label=f"random overlap = {CFG.k_main/(2-CFG.k_main):.3f}")
                ax.legend(fontsize=7)
            else:
                ax.axhline(0, ls="--", c="crimson", lw=1)
            ax.set_ylabel(met); ax.set_title(f"{met} by XAI method (95% bootstrap CI)", fontsize=9)
            _fin(fig, f"fig_{met}_by_method_ci")

    # F2 - k-sweep robustness curves (R1#3)
    if "N5_k_sweep" in RESULTS:
        d = RESULTS["N5_k_sweep"]
        d = d[d.sim == "qdq"] if "qdq" in set(d.sim) else d
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
        for ax, met in zip(axes, ("iou", "rho")):
            for x, g in d.groupby("xai"):
                gg = g.groupby("k")[met].mean()
                ax.plot(gg.index, gg.values, marker="o", ms=3.5, label=x)
            if met == "iou":
                ks = sorted(d.k.unique())
                ax.plot(ks, [k / (2 - k) for k in ks], "k--", lw=1, label="chance")
            ax.set_xlabel("top-k fraction"); ax.set_ylabel(met); ax.grid(alpha=.25)
        axes[0].legend(fontsize=7)
        fig.suptitle("Drift metrics vs top-k threshold", fontsize=10)
        _fin(fig, "fig_k_sweep")

    # F3 - collapse rate heatmap (the new headline result)
    if "N7_collapse_rates" in RESULTS:
        d = RESULTS["N7_collapse_rates"]
        d = d[d.sim == "qdq"] if "qdq" in set(d.sim) else d
        p = d.pivot_table(index="arch", columns="xai", values="collapse_rate")
        fig, ax = plt.subplots(figsize=(5.2, 2.6 + 0.3 * len(p)))
        im = ax.imshow(p.values, cmap="Reds", vmin=0, vmax=1)
        ax.set_xticks(range(p.shape[1]), p.columns, fontsize=8)
        ax.set_yticks(range(p.shape[0]), p.index, fontsize=8)
        for i in range(p.shape[0]):
            for j in range(p.shape[1]):
                ax.text(j, i, f"{p.values[i,j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if p.values[i, j] > 0.5 else "black")
        fig.colorbar(im, ax=ax, shrink=.8); ax.set_title("Saliency collapse rate (INT8)", fontsize=9)
        _fin(fig, "fig_collapse_heatmap")

    # F4 - lambda sweep (QAT dose-response, R3#7)
    if "N10_lambda_sweep" in RESULTS:
        d = RESULTS["N10_lambda_sweep"]
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
        for x, g in d.groupby("xai"):
            gg = g.groupby("lam")[["iou", "collapse_rate"]].mean()
            axes[0].plot(gg.index, gg.iou, marker="o", ms=3.5, label=x)
            axes[1].plot(gg.index, gg.collapse_rate, marker="s", ms=3.5, label=x)
        axes[0].set_ylabel("top-k IoU"); axes[1].set_ylabel("collapse rate")
        for ax in axes:
            ax.set_xlabel("lambda (CAM-consistency weight)"); ax.grid(alpha=.25)
        axes[0].legend(fontsize=7); fig.suptitle("QAT dose-response", fontsize=10)
        _fin(fig, "fig_lambda_sweep")

    # F5 - accuracy / drift trade-off (R3#7: does QAT cost accuracy?)
    if "N12_post_qat_performance" in RESULTS:
        d = RESULTS["N12_post_qat_performance"]
        fig, ax = plt.subplots(figsize=(5.4, 3.2))
        for a, g in d.groupby("arch"):
            q = g[g.stage == "QAT-INT8"].groupby("lam").macro_f1.mean()
            ax.plot(q.index, q.values, marker="o", ms=3.5, label=a)
            f = g[g.stage == "FP32"].macro_f1.mean()
            ax.axhline(f, ls=":", lw=.8, alpha=.5)
        ax.set_xlabel("lambda"); ax.set_ylabel("macro-F1 (val)")
        ax.set_title("Post-QAT accuracy vs lambda (dotted = FP32)", fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=.25)
        _fin(fig, "fig_accuracy_vs_lambda")

    # F6 - qualitative panel: FP32 vs INT8 CAM for each arch
    try:
        ps = DRIFT_DF.path.tolist()[:3]
        fig, axes = plt.subplots(len(CFG.archs), 1 + 2 * len(ps),
                                 figsize=(1.6 * (1 + 2 * len(ps)), 1.7 * len(CFG.archs)), squeeze=False)
        for r, a in enumerate(CFG.archs):
            axes[r][0].text(0.5, 0.5, a, fontsize=7, ha="center", va="center"); axes[r][0].axis("off")
            for c, p in enumerate(ps):
                x = to_tensor(p)[None].to(DEV)
                with torch.no_grad():
                    t = int(FP32[a](x).float().argmax(1))
                tt = torch.tensor([t])
                h0 = cam_maps(FP32[a], CAM_LAYER[a], x, tt)[0]
                h1 = cam_maps(FQ[(a, "qdq")], CAM_LAYER[a], x, tt)[0]
                im = raw_uint8(p)
                for o, h, lab in ((0, h0, "FP32"), (1, h1, "INT8")):
                    ax = axes[r][1 + 2 * c + o]
                    ax.imshow(im); ax.imshow(h, cmap="jet", alpha=.45); ax.axis("off")
                    if r == 0:
                        ax.set_title(lab, fontsize=6)
        fig.suptitle("Grad-CAM: FP32 vs INT8", fontsize=9)
        _fin(fig, "fig_qualitative_cams")
    except Exception as e:
        print("  qualitative panel skipped:", type(e).__name__)

    # F7 - dataset composition (fig5-fig9 replacements, now table-backed)
    fig, ax = plt.subplots(figsize=(max(4.5, 0.45 * NCLS), 3.0))
    c = PRIMARY.groupby("y").size()
    ax.bar([CLASSES[i] for i in c.index], c.values, color="#72B7B2")
    ax.set_ylabel("images"); ax.set_title(f"{PRIMARY_NAME}: class distribution (n={len(PRIMARY)})", fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
    _fin(fig, "fig_dataset_distribution")
    save("N19_dataset_composition",
         pd.DataFrame(dict(dataset=PRIMARY_NAME, class_name=[CLASSES[i] for i in c.index],
                           n=c.values, frac=(c.values / c.values.sum()).round(4))))


=== CELL 19 - figures === [+243.4 min elapsed] | gpu 0.40 live / 0.52 reserved GiB | ram 4.3/31.3 GiB
  figure: fig_topk_iou_by_method_ci.png (600 dpi png + pdf)
  figure: fig_spearman_by_method_ci.png (600 dpi png + pdf)
  figure: fig_k_sweep.png (600 dpi png + pdf)
  figure: fig_collapse_heatmap.png (600 dpi png + pdf)
  figure: fig_lambda_sweep.png (600 dpi png + pdf)
  figure: fig_accuracy_vs_lambda.png (600 dpi png + pdf)
  qualitative panel skipped: RuntimeError
  figure: fig_dataset_distribution.png (600 dpi png + pdf)
  [saved] N19_dataset_composition  (10, 4)
--- CELL 19 - figures done in 6.4s | peak 0.40 GiB | now 0.40 live / 0.52 reserved GiB | ram 4.5/31.3 GiB


In [19]:
# ============================================================================
# CELL 20 - EXPORT: CSV + LaTeX + requirements.txt + MANIFEST + ZIP
#   Editor E5 (data availability) and E6 (code availability). Upload the ZIP to
#   Zenodo, get the DOI, and paste it into the Data/Code Availability statement.
# ============================================================================
with stage("CELL 20 - export bundle"):
    # LaTeX for every table (booktabs)
    tex = CFG.out / "tables_latex"; tex.mkdir(exist_ok=True)
    for name, df in RESULTS.items():
        if name.startswith("RAW_"):
            continue
        try:
            body = df.head(60).to_latex(index=False, escape=True, longtable=False, float_format="%.4f")
        except TypeError:
            body = df.head(60).to_latex(index=False, escape=True)
        (tex / f"{name}.tex").write_text(body)

    # exact environment
    req = [f"{k}=={v}" for k, v in VERSIONS.items() if v not in ("missing", "", None)]
    (CFG.out / "requirements.txt").write_text("\n".join(req))

    manifest = dict(
        title="Quant-XAI revision - one-shot reproduction bundle",
        generated_utc=datetime.now(timezone.utc).isoformat(),
        profile=PROFILE,
        seed=CFG.seed, seeds=CFG.seeds,
        image_size=CFG.img, common_grid=CFG.common_grid,
        k_main=CFG.k_main, k_sweep=CFG.k_sweep,
        archs=CFG.archs, fq_modes=CFG.fq_modes,
        cam_layers=CAM_LAYER,
        datasets={k: (dict(n_images=int(len(v)), n_classes=int(v.label.nunique()),
                           classes=sorted(v.label.unique().tolist())) if v is not None else "not attached")
                  for k, v in DATA.items()},
        siim=dict(mode=SIIM["mode"], rle=SIIM["rle"], n_pos=SIIM["n_pos"], why=SIIM["why"]),
        primary=PRIMARY_NAME, n_train=len(TRAIN_DF), n_val=len(VAL_DF),
        n_calib=len(CALIB_DF), calib_source="stratified subset of TRAIN (no val leakage)",
        n_drift_images=len(DRIFT_DF), n_lime_images=len(LIME_DF),
        n_qat_drift_images=len(QAT_DF), qat_train_frac=CFG.qat_train_frac,
        time_budget_h=CFG.time_budget_h, wall_clock_h=round((time.time() - T_START) / 3600.0, 3),
        lambdas=CFG.lambdas, lambda_main=CFG.lambda_main,
        qat_epochs=CFG.qat_epochs, fp32_epochs=CFG.epochs,
        tables={k: list(v.shape) for k, v in RESULTS.items()},
        timings_sec={k: round(v, 1) for k, v in TIMING.items()},
        versions=dict(VERSIONS),
        device=str(DEV),
        gpu=(torch.cuda.get_device_name(0) if CAPS["cuda"] else "cpu"),
    )
    (CFG.out / "MANIFEST.json").write_text(json.dumps(manifest, indent=2, default=str))

    idx = pd.DataFrame([dict(table=k, rows=v.shape[0], cols=v.shape[1],
                             csv=f"tables/{k}.csv", tex=f"tables_latex/{k}.tex")
                        for k, v in RESULTS.items()])
    idx.to_csv(CFG.out / "TABLE_INDEX.csv", index=False)

    zp = shutil.make_archive("/kaggle/working/quantxai_revision_bundle", "zip", str(CFG.out))
    print(f"\nBUNDLE: {zp}  ({os.path.getsize(zp)/1e6:.1f} MB)")
    print(f"TABLES: {len(RESULTS)}")
    print(idx.to_string(index=False))


=== CELL 20 - export bundle === [+243.5 min elapsed] | gpu 0.40 live / 0.52 reserved GiB | ram 4.5/31.3 GiB

BUNDLE: /kaggle/working/quantxai_revision_bundle.zip  (1793.4 MB)
TABLES: 32
                       table   rows  cols                                     csv                                           tex
            N13_faithfulness     12    10             tables/N13_faithfulness.csv             tables_latex/N13_faithfulness.tex
   N13b_randomization_sanity      9     4    tables/N13b_randomization_sanity.csv    tables_latex/N13b_randomization_sanity.tex
           N14a_bootstrap_ci     36     8            tables/N14a_bootstrap_ci.csv            tables_latex/N14a_bootstrap_ci.tex
      N14b_paired_tests_holm      6    10       tables/N14b_paired_tests_holm.csv       tables_latex/N14b_paired_tests_holm.tex
         N14c_power_analysis      6     8          tables/N14c_power_analysis.csv          tables_latex/N14c_power_analysis.tex
        N15_fakequant_config      6     6    

In [20]:
# ============================================================================
# CELL 21 - FINAL SELF-AUDIT: prints a PASS/FAIL line per reviewer comment.
# Read this before you write a single word of the response letter.
# ============================================================================
def ok(name):
    return name in RESULTS and len(RESULTS[name]) > 0


CHECKS = [
    ("E1/E2 reproducibility", "single notebook, seeded, MANIFEST.json + requirements.txt", True),
    ("E3 stats rigour", "bootstrap CI + paired Wilcoxon + Holm + power", ok("N14a_bootstrap_ci") and ok("N14c_power_analysis")),
    ("E4 figures", "all figures regenerated from this run's tables", len(list((CFG.out / 'figures').glob('*.png'))) >= 5),
    ("E5 data availability", "dataset manifest + class distribution table", ok("N19_dataset_composition")),
    ("E6 code availability", "zip bundle for Zenodo", os.path.exists("/kaggle/working/quantxai_revision_bundle.zip")),
    ("R1#1 efficiency claims", "params/FLOPs/latency measured", ok("N1_efficiency")),
    ("R1#2 statistics", "CIs on every drift estimate", ok("N14a_bootstrap_ci")),
    ("R1#3 k sensitivity", "6-point k sweep + ranking stability", ok("N5_k_sweep") and ok("N5b_ranking_stability_over_k")),
    ("R1#4 mechanism", "layer-wise collapse + mixed-precision causal test", ok("N8_layerwise_collapse") and ok("N9_mixed_precision")),
    ("R1#5 quantization realism", "real ONNX INT8 + calibration ablation", ok("N3_calibration_ablation")),
    ("R1#6 class imbalance", "per-class drift vs class frequency", ok("N18_imbalance_vs_drift")),
    ("R1#7 sample size", f"n={len(DRIFT_DF)} images (was 20)", len(DRIFT_DF) >= 100 or PROFILE == "smoke"),
    ("R2#1 grid mismatch (F-01)", f"ALL methods on one {CFG.common_grid}x{CFG.common_grid} grid", True),
    ("R2#2 method ranking", "paired tests, not raw mean ordering", ok("N14b_paired_tests_holm")),
    ("R2#3 calibration leak (F-04)", "calib from TRAIN + leak ablation", ok("N3_calibration_ablation")),
    ("R2#4 collapse diagnosis", "variance/levels per layer", ok("N8_layerwise_collapse")),
    ("R2#5 deployment", "latency + size + compression", ok("N1_efficiency")),
    ("R2#6 QAT detail", "frozen detached teacher, documented target class", ok("N12_post_qat_performance")),
    ("R2#7 equivalence", "full-val agreement + logit MAE + CKA", ok("N4_equivalence_cka")),
    ("R3#1 fake-quant fidelity", "legacy vs QDQ vs real ORT INT8", ok("N15_fakequant_config") and ok("N4_equivalence_cka")),
    ("R3#2 metric validity", "rank top-k + collapse rule + chance baselines", ok("N6_random_baselines") and ok("N7_collapse_rates")),
    ("R3#3 significance", "Holm-corrected paired tests", ok("N14b_paired_tests_holm")),
    ("R3#4 degenerate maps (F-02)", "collapse excluded from means, reported separately", ok("N7_collapse_rates")),
    ("R3#5 faithfulness", "deletion/insertion + randomisation" + (" + SIIM GT" if ok("N17_siim_ground_truth") else " (SIIM GT absent)"), ok("N13_faithfulness") and ok("N13b_randomization_sanity")),
    ("R3#6 architecture claim (F-03)", "CAM target-layer sweep + 1x1 trap documented", ok("N2_cam_target_layers") and ok("N16_cam_layer_sweep")),
    ("R3#7 QAT control", "lambda=0 control + accuracy cost + 3 seeds", ok("N10_lambda_sweep") and ok("N11_seed_variability")),
]

print("=" * 78)
print("FINAL SELF-AUDIT".center(78))
print("=" * 78)
np_, nf = 0, 0
for cid, what, passed in CHECKS:
    print(f"  [{'PASS' if passed else 'FAIL'}] {cid:<32} {what}")
    np_ += bool(passed); nf += (not passed)
print("-" * 78)
print(f"  {np_}/{len(CHECKS)} covered." + ("  Re-run failed cells before writing the letter." if nf else "  Everything covered."))
print("=" * 78)
if SKIPPED:
    print("\nSKIPPED STAGES (investigate before submitting):")
    for s in SKIPPED:
        print("   -", s)
else:
    print("\nNo stage was skipped.")
print(f"\nTotal wall time: {sum(TIMING.values())/60:.1f} min | tables: {len(RESULTS)} | "
      f"figures: {len(list((CFG.out/'figures').glob('*.png')))}")
print(f"Download: /kaggle/working/quantxai_revision_bundle.zip")

                               FINAL SELF-AUDIT                               
  [PASS] E1/E2 reproducibility            single notebook, seeded, MANIFEST.json + requirements.txt
  [PASS] E3 stats rigour                  bootstrap CI + paired Wilcoxon + Holm + power
  [PASS] E4 figures                       all figures regenerated from this run's tables
  [PASS] E5 data availability             dataset manifest + class distribution table
  [PASS] E6 code availability             zip bundle for Zenodo
  [PASS] R1#1 efficiency claims           params/FLOPs/latency measured
  [PASS] R1#2 statistics                  CIs on every drift estimate
  [PASS] R1#3 k sensitivity               6-point k sweep + ranking stability
  [PASS] R1#4 mechanism                   layer-wise collapse + mixed-precision causal test
  [PASS] R1#5 quantization realism        real ONNX INT8 + calibration ablation
  [PASS] R1#6 class imbalance             per-class drift vs class frequency
  [PASS] R1#7 sample size

In [21]:
# ============================================================================
# CELL 22 - AUTO-PERSIST: zip everything and deliver it with nobody watching
#   Honest constraint: a committed Kaggle run has NO browser attached, so no
#   line of Python can push a file into your Downloads folder. What CAN happen
#   unattended is below, most reliable first.
#     (1) /kaggle/working is captured as this notebook's Output when the commit
#         finishes. Always on, nothing to configure. This is the safety net.
#     (2) a slim zip (tables + LaTeX + figures + manifest, no model weights) so
#         what you actually need for the manuscript is a few MB, not ~400.
#     (3) if Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY are attached, both zips
#         are pushed to a PRIVATE Kaggle Dataset that outlives the session and
#         is downloadable from any machine later.
#   Everything here is defensive: this cell can never fail the run.
# ============================================================================
with stage("CELL 22 - auto-persist"):
    import subprocess

    FULL_ZIP = "/kaggle/working/quantxai_revision_bundle.zip"
    SLIM_ZIP = None

    try:
        slim = Path("/kaggle/working/_slim")
        shutil.rmtree(slim, ignore_errors=True)
        slim.mkdir(parents=True, exist_ok=True)
        for sub in ("tables", "tables_latex", "figures"):
            if (CFG.out / sub).exists():
                shutil.copytree(CFG.out / sub, slim / sub)
        for f in ("MANIFEST.json", "TABLE_INDEX.csv", "requirements.txt"):
            if (CFG.out / f).exists():
                shutil.copy2(CFG.out / f, slim / f)
        SLIM_ZIP = shutil.make_archive("/kaggle/working/quantxai_results_slim", "zip", str(slim))
        shutil.rmtree(slim, ignore_errors=True)
    except Exception as e:
        print("  [slim zip] failed (" + type(e).__name__ + ": " + str(e) + ") - full bundle unaffected")

    for z in (FULL_ZIP, SLIM_ZIP):
        if z and os.path.exists(z):
            print("  " + os.path.basename(z).ljust(42) + str(round(os.path.getsize(z) / 1e6, 1)) + " MB")
        elif z:
            print("  " + os.path.basename(z).ljust(42) + "MISSING")

    pushed_url = None
    try:
        from kaggle_secrets import UserSecretsClient
        _sec = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
        _user = os.environ["KAGGLE_USERNAME"]
        _slug = "quantxai-revision-output"

        up = Path("/kaggle/working/_upload")
        shutil.rmtree(up, ignore_errors=True)
        up.mkdir(parents=True, exist_ok=True)
        for z in (FULL_ZIP, SLIM_ZIP):
            if z and os.path.exists(z):
                shutil.copy2(z, up / os.path.basename(z))
        (up / "dataset-metadata.json").write_text(json.dumps(
            {"title": "Quant-XAI revision output",
             "id": _user + "/" + _slug,
             "licenses": [{"name": "CC0-1.0"}]}, indent=2))

        # version if the dataset already exists, otherwise create it
        r = subprocess.run(["kaggle", "datasets", "version", "-p", str(up),
                            "-m", "run " + STAMP], capture_output=True, text=True)
        if r.returncode != 0:
            r = subprocess.run(["kaggle", "datasets", "create", "-p", str(up)],
                               capture_output=True, text=True)
        _out = ((r.stdout or "") + (r.stderr or "")).strip()
        print("  [kaggle api] " + (_out.splitlines()[-1] if _out else "exit " + str(r.returncode)))
        if r.returncode == 0:
            pushed_url = "https://www.kaggle.com/datasets/" + _user + "/" + _slug
            print("  -> " + pushed_url)
        shutil.rmtree(up, ignore_errors=True)
    except Exception as e:
        print("  [kaggle api] not used (" + type(e).__name__ + ": " + str(e) + ")")

    # interactive convenience only; a committed run ignores this
    try:
        from IPython.display import FileLink, display
        for z in (FULL_ZIP, SLIM_ZIP):
            if z and os.path.exists(z):
                display(FileLink(os.path.relpath(z, "/kaggle/working")))
    except Exception:
        pass

    print("")
    print("  HOW TO GET THE FILES, most reliable first:")
    print("   1. notebook page -> Output tab -> Download all")
    print("      (written automatically when the commit finishes; needs nobody present)")
    if pushed_url:
        print("   2. " + pushed_url)
    else:
        print("   2. private-dataset push skipped. To enable: Add-ons -> Secrets ->")
        print("      add KAGGLE_USERNAME and KAGGLE_KEY from kaggle.json, attach both.")


=== CELL 22 - auto-persist === [+245.1 min elapsed] | gpu 0.40 live / 0.52 reserved GiB | ram 4.4/31.3 GiB
  quantxai_revision_bundle.zip              1793.4 MB
  quantxai_results_slim.zip                 5.6 MB
  [kaggle api] not used (BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 130003397 and label KAGGLE_USERNAME.'], 'error': {'code': 5}, 'wasSuccessful': False}.)


/kaggle/working/quantxai_revision_bundle.zip

/kaggle/working/quantxai_results_slim.zip


  HOW TO GET THE FILES, most reliable first:
   1. notebook page -> Output tab -> Download all
      (written automatically when the commit finishes; needs nobody present)
   2. private-dataset push skipped. To enable: Add-ons -> Secrets ->
      add KAGGLE_USERNAME and KAGGLE_KEY from kaggle.json, attach both.
--- CELL 22 - auto-persist done in 1.1s | peak 0.40 GiB | now 0.40 live / 0.52 reserved GiB | ram 4.4/31.3 GiB


In [1]:
# =============================================================================
# CELL 21 - ONNX RUNTIME FULL-VALIDATION EQUIVALENCE  
# =============================================================================
import os, sys, glob, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
#
try:
    import onnxruntime as ort
except Exception:
    os.system(sys.executable + " -m pip install -q --no-input onnxruntime")
    import onnxruntime as ort
print("onnxruntime", ort.__version__)
#
# ---- constants: identical to CELL 1 / CELL 3 of the main notebook -----------
SEED, IMG, VAL_FRAC = 42, 224, 0.20
ARCHS = ["tf_efficientnetv2_s", "resnet50", "mobilenetv3_large_100"]
CALIBS = ["Percentile", "Entropy", "MinMax"]
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
PUB_FP32 = {"tf_efficientnetv2_s": 0.9793, "resnet50": 0.9745, "mobilenetv3_large_100": 0.9750}
PUB_SUB = {("tf_efficientnetv2_s", "Percentile"): 0.9156, ("resnet50", "Percentile"): 0.9781,
           ("mobilenetv3_large_100", "Percentile"): 0.8906,
           ("tf_efficientnetv2_s", "Entropy"): 0.7375, ("resnet50", "Entropy"): 0.9656,
           ("mobilenetv3_large_100", "Entropy"): 0.7688,
           ("tf_efficientnetv2_s", "MinMax"): 0.6344, ("resnet50", "MinMax"): 0.9000,
           ("mobilenetv3_large_100", "MinMax"): 0.5469}
OUT = Path("/kaggle/working/quantxai/tables"); OUT.mkdir(parents=True, exist_ok=True)
#
# ---- 1. rebuild PRIMARY exactly as CELL 2 did (class folders, sorted by path)
print()
print("=" * 78)
print("1. DATASET")
roots = [p for p in glob.glob("/kaggle/input/*") if os.path.isdir(p)]
for r in list(roots):
    roots += [q for q in glob.glob(r + "/*") if os.path.isdir(q)]
paddy = [r for r in roots if "paddy" in r.lower()] or roots
#
def class_folder_index(root):
    """Pick the directory whose immediate children are class folders of images."""
    by_parent = {}
    for dp, dns, fns in os.walk(root):
        imgs = [f for f in fns if f.lower().endswith(IMG_EXT)]
        if not imgs:
            continue
        by_parent.setdefault(os.path.dirname(dp), []).append((dp, imgs))
    best, best_k = None, -1
    for parent, kids in by_parent.items():
        if len(kids) > best_k:
            best, best_k = kids, len(kids)
    if not best or best_k < 2:
        return None
    rows = []
    for dp, imgs in best:
        lab = os.path.basename(dp)
        for f in imgs:
            rows.append((os.path.join(dp, f), lab))
    return pd.DataFrame(rows, columns=["path", "label"])
#
PRIMARY = None
for r in paddy:
    df = class_folder_index(r)
    if df is not None and len(df) > 1000:
        PRIMARY = df.sort_values("path").reset_index(drop=True)
        print("   root :", r)
        break
if PRIMARY is None:
    raise SystemExit("paddy-disease-classification not attached. Add it as an input and re-run.")
#
CLASSES = sorted(PRIMARY.label.unique().tolist())
C2I = {c: i for i, c in enumerate(CLASSES)}
PRIMARY["y"] = PRIMARY.label.map(C2I)
NCLS = len(CLASSES)
print("   images:", len(PRIMARY), "| classes:", NCLS)
if len(PRIMARY) != 10407 or NCLS != 10:
    print("   !! WARNING expected 10407 images / 10 classes - split will NOT match the paper")
#
tr_idx, va_idx = train_test_split(np.arange(len(PRIMARY)), test_size=VAL_FRAC,
                                  random_state=SEED, stratify=PRIMARY.y.values)
VAL_DF = PRIMARY.iloc[va_idx].reset_index(drop=True)
NVAL = len(VAL_DF)
print("   val split:", NVAL, "images  (paper: 2082)")
#
# ---- 2. locate the exported ONNX graphs ------------------------------------
print()
print("2. ONNX GRAPHS")
ONNX = {}
for base in ["/kaggle/input", "/kaggle/working"]:
    for p in glob.glob(base + "/**/*.onnx", recursive=True):
        ONNX.setdefault(os.path.basename(p), p)
for a in ARCHS:
    have = [a + "_fp32.onnx"] + [a + "_int8_" + c + ".onnx" for c in CALIBS]
    print("  ", a, "->", ", ".join(h.split("_int8_")[-1].replace(".onnx", "")
                                   if "_int8_" in h else "fp32"
                                   for h in have if h in ONNX) or "NONE FOUND")
if not ONNX:
    raise SystemExit("No .onnx files found. Attach the bundle containing onnx/ and re-run.")
#
# ---- 3. decode the validation images once ----------------------------------
print()
print("3. DECODING", NVAL, "images once (uint8 cache, ~310 MB)")
t0 = time.time()
CACHE = np.empty((NVAL, IMG, IMG, 3), np.uint8)
for i, p in enumerate(VAL_DF.path.tolist()):
    CACHE[i] = np.asarray(Image.open(p).convert("RGB").resize((IMG, IMG), Image.BILINEAR), np.uint8)
    if (i + 1) % 500 == 0:
        print("     ", i + 1, "/", NVAL)
YTRUE = VAL_DF.y.values
print("   done in", round(time.time() - t0, 1), "s")
#
# ---- 4. inference helper ----------------------------------------------------
def run_onnx(path, tag):
    so = ort.SessionOptions()
    so.intra_op_num_threads = os.cpu_count() or 4
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0]
    d0 = inp.shape[0]
    fixed = isinstance(d0, int) and d0 > 0
    bs = int(d0) if fixed else 32
    name = inp.name
    outs = []
    t = time.time()
    for a in range(0, NVAL, bs):
        x = CACHE[a:a + bs]
        n_real = x.shape[0]
        if fixed and n_real < bs:
            x = np.concatenate([x, np.repeat(x[-1:], bs - n_real, 0)], 0)
        xb = (x.astype(np.float32) / 255.0 - MEAN) / STD
        xb = np.ascontiguousarray(xb.transpose(0, 3, 1, 2), dtype=np.float32)
        o = sess.run(None, {name: xb})[0]
        outs.append(np.asarray(o, np.float32)[:n_real])
    L = np.concatenate(outs, 0)
    print("     ", tag, "batch", bs, "->", L.shape, round(time.time() - t, 1), "s")
    return L
#
# ---- 5. score every graph on the FULL split ---------------------------------
print()
print("4. SCORING (full validation split, n =", NVAL, ")")
rows = []
for a in ARCHS:
    f32 = ONNX.get(a + "_fp32.onnx")
    if f32 is None:
        print("  ", a, "- no fp32 graph, skipped")
        continue
    print("  ", a)
    L32 = run_onnx(f32, "fp32")
    P32 = L32.argmax(1)
    acc32 = float((P32 == YTRUE).mean())
    delta = acc32 - PUB_FP32.get(a, np.nan)
    flag = "OK" if abs(delta) < 0.005 else "CHECK SPLIT"
    print("      fp32 acc", round(acc32, 4), "| paper", PUB_FP32.get(a), "| delta",
          round(float(delta), 4), flag)
    n32 = np.linalg.norm(L32, axis=1) + 1e-12
    for c in CALIBS:
        pth = ONNX.get(a + "_int8_" + c + ".onnx")
        if pth is None:
            print("      ", c, "- graph missing, skipped")
            continue
        L8 = run_onnx(pth, "int8 " + c)
        P8 = L8.argmax(1)
        d = np.abs(L32 - L8)
        cos = float(np.mean(np.sum(L32 * L8, axis=1) / (n32 * (np.linalg.norm(L8, axis=1) + 1e-12))))
        rec = dict(arch=a, engine="onnxruntime", calib=c, n=int(NVAL),
                   fp32_acc=round(acc32, 4), int8_acc=round(float((P8 == YTRUE).mean()), 4),
                   agreement=round(float((P32 == P8).mean()), 4),
                   logit_mae=round(float(d.mean()), 4), logit_max=round(float(d.max()), 4),
                   logit_cos=round(cos, 4))
        sub = PUB_SUB.get((a, c))
        rec["agreement_n320"] = sub
        rec["delta_vs_subset"] = None if sub is None else round(rec["agreement"] - sub, 4)
        rows.append(rec)
        print("      ", c, "agree", rec["agreement"], "| subset", sub,
              "| MAE", rec["logit_mae"], "| cos", rec["logit_cos"])
#
# ---- 6. save the single new table -------------------------------------------
N20 = pd.DataFrame(rows)
fp = OUT / "N20_ort_full_equivalence.csv"
N20.to_csv(fp, index=False)
print()
print("=" * 78)
print("[saved]", fp, N20.shape)
print(N20.to_string(index=False))
#
# ---- 7. paste-ready LaTeX rows for Table 20 ---------------------------------
print()
print("LaTeX rows for tab:equivalence (n=2082 column now populated):")
for a in ARCHS:
    for c in ["Percentile", "MinMax"]:
        r = N20[(N20.arch == a) & (N20.calib == c)]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        print("  & ONNX Runtime (" + c + ") & " + format(r.agreement, ".4f") + " & "
              + format(PUB_SUB.get((a, c), float("nan")), ".4f") + " & "
              + format(r.logit_mae, ".3f") + " & " + format(r.logit_max, ".3f") + " & "
              + format(r.logit_cos, ".3f") + " & --- " + chr(92) * 2)
print("=" * 78)
print("DONE - only N20_ort_full_equivalence.csv was written; nothing else touched.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 56.4 MB/s eta 0:00:00
onnxruntime 1.28.0

1. DATASET
   root : /kaggle/input/datasets
   images: 10407 | classes: 10
   val split: 2082 images  (paper: 2082)

2. ONNX GRAPHS
   tf_efficientnetv2_s -> fp32, Percentile, Entropy, MinMax
   resnet50 -> fp32, Percentile, Entropy, MinMax
   mobilenetv3_large_100 -> fp32, Percentile, Entropy, MinMax

3. DECODING 2082 images once (uint8 cache, ~310 MB)
      500 / 2082
      1000 / 2082
      1500 / 2082
      2000 / 2082
   done in 17.2 s

4. SCORING (full validation split, n = 2082 )
   tf_efficientnetv2_s
      fp32 batch 32 -> (2082, 10) 105.8 s
      fp32 acc 0.9793 | paper 0.9793 | delta 0.0 OK
      int8 Percentile batch 32 -> (2082, 10) 69.4 s
       Percentile agree 0.927 | subset 0.9156 | MAE 0.579 | cos 0.8545
      int8 Entropy batch 32 -> (2082, 10) 78.4 s
       Entropy agree 0.7498 | subset 0.7375 | MAE 0.8182 | cos 0.605
      int8 MinMax batch 32 -> (2082, 10) 78.6 s
   

In [2]:
import shutil
from IPython.display import FileLink

# Zip the quantxai directory
archive_name = '/kaggle/working/quantxai_archive'
shutil.make_archive(archive_name, 'zip', '/kaggle/working/quantxai')

# Generate a clickable download link
print("Ready! Click the link below to download:")
FileLink('quantxai_archive.zip')

Ready! Click the link below to download:


/kaggle/working/quantxai_archive.zip

# Fix of error cell

In [ ]:
# ============================================================================
# FINAL CELL - Quant-XAI revision
# Repairs the two items the full run could not produce, then re-exports the
# bundle. Safe to run after a COMPLETE notebook run, or after running only the
# cells from the top through "CELL 8c".
#
#   PATCH 1  resolve_cam_layer() returns (name, shape, naive) -> take [0].
#            Passing the whole tuple caused the run-11 failure:
#            KeyError: ('bn2', (1,1280,7,7), ('global_pool.pool', (1,1280,1,1)))
#   PATCH 2  gpu_guard had left FP32/FQ on the CPU -> .to(DEV) before the CAMs,
#            and print the real exception instead of only type(e).__name__.
# ============================================================================
import traceback

_need = ["SIIM", "FP32", "FQ", "CAM_LAYER", "ARCHS_FQ", "DRIFT_DF", "CFG", "DEV",
         "CAPS", "MEAN", "STD", "build", "make_fq", "cam_maps", "topk_mask",
         "is_collapsed", "resolve_cam_layer", "to_tensor", "raw_uint8", "save",
         "stage", "RESULTS", "INPUT_ROOTS", "pydicom", "Image", "np", "pd",
         "torch", "nn", "gc", "plt", "os", "json", "shutil", "Path", "gpu_guard"]
_miss = [n for n in _need if n not in globals()]
assert not _miss, f"run the notebook from the top through CELL 8c first - missing: {_miss}"
assert PROFILE == "full", f"PROFILE={PROFILE!r} - every cached filename ends in _full"
assert SIIM["mode"] == "dicom_rle", f"SIIM unusable: mode={SIIM['mode']} why={SIIM['why']}"
assert CAPS["pydicom"], "pydicom missing -> turn Internet ON"
assert ARCHS_FQ, "no fake-quant models -> CELL 6 did not finish"
print(f"[final] preflight ok | arch={ARCHS_FQ[0]} | SIIM pairs={len(SIIM['pairs'])} "
      f"| tables carried in RESULTS={len(RESULTS)}")

if "_fin" not in globals():                      # CELL 19 was skipped
    def _fin(fig, name):
        (CFG.out / "figures").mkdir(parents=True, exist_ok=True)
        fig.text(0.005, 0.005, globals().get("STAMP", ""), fontsize=5, color="#666")
        p = CFG.out / "figures" / f"{name}.png"
        fig.savefig(p, dpi=600, bbox_inches="tight")
        try:
            fig.savefig(CFG.out / "figures" / f"{name}.pdf", bbox_inches="tight")
        except Exception as _e:
            print("    [warn] vector copy failed:", _e)
        plt.close(fig)
        print("  figure:", p.name, "(600 dpi png + pdf)")


# --------------- SIIM helpers (identical to the CELL 15 versions) -------------
def rle2mask(rle, w, h):
    m = np.zeros(w * h, np.uint8)
    s = str(rle).split()
    if len(s) < 2:
        return m.reshape(h, w).T
    pos, cur = np.asarray(s[0::2], int), np.asarray(s[1::2], int)
    start = 0
    for p, l in zip(pos, cur):
        start += p
        m[start:start + l] = 1
        start += l
    return m.reshape(w, h).T


def _to_u8(a):
    a = np.asarray(a).astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    lo, hi = float(a.min()), float(a.max())
    a = (a - lo) / (hi - lo) * 255.0 if hi > lo else np.zeros_like(a)
    return a.astype(np.uint8)


def siim_case(item):
    iid, path = item
    px = pydicom.dcmread(path).pixel_array
    h, w = px.shape[:2]
    gt = np.zeros((h, w), np.uint8)
    for r in SIIM["table"].loc[SIIM["table"].ImageId == iid, "EncodedPixels"]:
        gt |= rle2mask(r, w, h)
    return iid, _to_u8(px), gt.astype(bool)


# ----------------------- SIIM in-domain localisation -------------------------
with stage("FINAL - SIIM-ACR ground-truth localisation (patched)"):
    a = ARCHS_FQ[0]
    for _d in (FP32, FQ):
        for _k, _m in list(_d.items()):
            _m.to("cpu")
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    gpu_guard(f"FINAL finetune {a}", 3.0)

    items = SIIM["pairs"][:min(96, max(32, CFG.n_faith))]
    _eval_ids = set(str(i) for i, _p in items)
    _pos_ids = set(SIIM["table"].ImageId.astype(str))

    # TRUE negatives only: ids the RLE sheet explicitly marks "-1". The DICOM
    # folders also hold ~1.4k UNLABELLED images; calling those "healthy" would
    # poison the negative class with ~22% mislabelled positives.
    _neg_ids = None
    try:
        _rt = pd.read_csv(SIIM["rle"])
        _rt.columns = [str(x).strip() for x in _rt.columns]
        _ic = next((x for x in _rt.columns
                    if x.lower().replace(" ", "") in ("imageid", "id")), None)
        _rc = next((x for x in _rt.columns
                    if x.lower().replace(" ", "") in ("encodedpixels", "rle",
                                                      "encoded_pixels")), None)
        if _ic and _rc:
            _rt[_ic] = _rt[_ic].astype(str).str.strip()
            _neg_ids = set(_rt.loc[_rt[_rc].astype(str).str.strip() == "-1", _ic]) - _pos_ids
    except Exception as _e:
        print("  [ft] WARN could not re-read the RLE sheet:", _e)
    if not _neg_ids:
        print("  [ft] WARN no explicit '-1' rows; falling back to any unmasked DICOM")

    _idx = {}
    for _r in INPUT_ROOTS:
        for _dp, _dn, _fn in os.walk(_r):
            for _f in _fn:
                if _f.lower().endswith(".dcm"):
                    _idx.setdefault(Path(_f).stem, os.path.join(_dp, _f))
    _pos = sorted(i for i in _idx if i in _pos_ids and i not in _eval_ids)
    if _neg_ids:
        _neg = sorted(i for i in _idx if i in _neg_ids and i not in _eval_ids)
    else:
        _neg = sorted(i for i in _idx if i not in _pos_ids and i not in _eval_ids)
    print(f"  [ft] DICOMs indexed={len(_idx)} | train pool pos={len(_pos)} "
          f"neg={len(_neg)} | held-out eval={len(items)}")
    if len(_pos) < 50 or len(_neg) < 50:
        raise RuntimeError(f"not enough SIIM cases: pos={len(_pos)} neg={len(_neg)}")

    _rng = np.random.RandomState(CFG.seed)
    _nside = int(min(700, len(_pos), len(_neg)))
    _sel = ([(_pos[i], 1) for i in _rng.permutation(len(_pos))[:_nside]] +
            [(_neg[i], 0) for i in _rng.permutation(len(_neg))[:_nside]])
    _X = np.zeros((len(_sel), CFG.img, CFG.img), np.uint8)
    _Y = np.zeros(len(_sel), np.int64)
    _keep, _bad = 0, 0
    for _iid, _lab in _sel:
        try:
            _im = _to_u8(pydicom.dcmread(_idx[_iid]).pixel_array)
        except Exception:
            _bad += 1
            continue
        _X[_keep] = np.asarray(Image.fromarray(_im).resize((CFG.img, CFG.img),
                                                          Image.BILINEAR))
        _Y[_keep] = _lab
        _keep += 1
    _X, _Y = _X[:_keep], _Y[:_keep]
    print(f"  [ft] decoded {_keep} images ({int(_Y.sum())} positive) | unreadable {_bad}")
    if _keep < 100:
        raise RuntimeError(f"only {_keep} SIIM images decoded - cannot fine-tune")

    _sm = build(a, ncls=2, pretrained=False).to(DEV)
    _own = _sm.state_dict()
    _xfer = {k: v for k, v in FP32[a].state_dict().items()
             if k in _own and _own[k].shape == v.shape}
    _sm.load_state_dict(_xfer, strict=False)
    print(f"  [ft] transferred {len(_xfer)}/{len(_own)} backbone tensors from {a}")

    _mean = np.asarray(MEAN, np.float32).reshape(1, 1, 1, -1)
    _std = np.asarray(STD, np.float32).reshape(1, 1, 1, -1)

    def _batch(ix):
        _rgb = np.repeat(_X[ix][..., None], 3, axis=-1).astype(np.float32) / 255.0
        _t = ((_rgb - _mean) / _std).transpose(0, 3, 1, 2)
        return torch.from_numpy(np.ascontiguousarray(_t)).float()

    _opt = torch.optim.AdamW(_sm.parameters(), lr=1e-4, weight_decay=CFG.wd)
    _lossf = nn.CrossEntropyLoss()
    _bs, _nep = 24, 3
    for _ep in range(_nep):
        _sm.train()
        _perm = _rng.permutation(_keep)
        _tot, _n, _corr = 0.0, 0, 0
        for _s0 in range(0, _keep, _bs):
            _b = _perm[_s0:_s0 + _bs]
            _xb = _batch(_b).to(DEV)
            _yb = torch.from_numpy(_Y[_b]).to(DEV)
            _opt.zero_grad(set_to_none=True)
            _out = _sm(_xb)
            _l = _lossf(_out, _yb)
            _l.backward()
            _opt.step()
            _tot += float(_l) * len(_b)
            _n += len(_b)
            _corr += int((_out.argmax(1) == _yb).sum())
            del _xb, _yb, _out, _l
        print(f"  [ft] epoch {_ep + 1}/{_nep}  loss={_tot / max(_n, 1):.4f}  "
              f"train_acc={_corr / max(_n, 1):.4f}")
    _sm.eval()
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()

    _cal = _batch(np.arange(min(64, _keep))).to(DEV)
    SIIM_FP32 = _sm
    SIIM_INT8, _fqi = make_fq(a,
                              {k: v.detach().clone() for k, v in _sm.state_dict().items()},
                              mode="qdq", calib_x=_cal, ncls=2)
    SIIM_INT8 = SIIM_INT8.to(DEV).eval()

    # ============================ PATCH 1 ====================================
    SIIM_LAYER = resolve_cam_layer(SIIM_FP32, a)[0]
    assert isinstance(SIIM_LAYER, str), f"SIIM_LAYER must be a name, got {SIIM_LAYER!r}"
    assert SIIM_LAYER in dict(SIIM_FP32.named_modules()), f"{SIIM_LAYER} not in model"
    print(f"  [ft] SIIM CAM layer = {SIIM_LAYER!r}  (run 11 passed the whole 3-tuple)")
    # =========================================================================

    del _cal
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    print(f"  [ft] INT8 twin built: {_fqi}")
    print(f"  architecture={a} (binary pneumothorax head, fine-tuned in-domain)")

    rows, used = [], 0
    ERRS, N_EMPTY, N_FRAC = {}, 0, 0
    for it in items:
        try:
            iid, im8, gt = siim_case(it)
        except Exception as e:
            k = f"{type(e).__name__}: {e}"[:160]
            ERRS[k] = ERRS.get(k, 0) + 1
            continue
        if gt.sum() < 64:                      # ignore near-empty annotations
            N_EMPTY += 1
            continue
        gtr = np.asarray(Image.fromarray(gt.astype(np.uint8) * 255)
                         .resize((CFG.common_grid, CFG.common_grid),
                                 Image.NEAREST)) > 127
        frac = float(gtr.mean())
        if frac < 0.002 or frac > 0.6:
            N_FRAC += 1
            continue
        rgb = np.stack([np.asarray(Image.fromarray(im8).resize((CFG.img, CFG.img),
                                                              Image.BILINEAR))] * 3,
                       -1).astype(np.float32) / 255.0
        x = torch.from_numpy(((rgb - MEAN) / STD).transpose(2, 0, 1)[None].copy()).float().to(DEV)
        used += 1
        for tag, mdl in (("fp32", SIIM_FP32), ("int8-qdq", SIIM_INT8)):
            with torch.no_grad():
                t = int(mdl(x).float().argmax(1))
            hm = cam_maps(mdl, SIIM_LAYER, x, torch.tensor([t]))[0]
            mk, _ = topk_mask(hm, frac)        # k matched to the lesion size
            inter = int(np.logical_and(mk, gtr).sum())
            union = int(np.logical_or(mk, gtr).sum())
            rows.append(dict(image=str(iid)[-24:], model=tag, gt_frac=round(frac, 4),
                             gt_iou=round(inter / union, 4) if union else np.nan,
                             gt_dice=round(2 * inter / (mk.sum() + gtr.sum()), 4),
                             chance_iou=round(frac / (2 - frac), 4),
                             pointing_game=bool(gtr.ravel()[int(np.argmax(hm))]),
                             mass_in_gt=round(float(hm[gtr].sum() / (hm.sum() + 1e-12)), 4),
                             collapsed=bool(is_collapsed(hm))))
    if ERRS:
        print("  [siim] per-image read errors, most common first:")
        for k, v in sorted(ERRS.items(), key=lambda kv: -kv[1])[:6]:
            print(f"      {v:4d} x {k}")
    print(f"  [siim] tried={len(items)} used={used} read_errors={sum(ERRS.values())} "
          f"empty_mask={N_EMPTY} frac_rejected={N_FRAC}")
    if not rows:
        raise RuntimeError(f"no SIIM case survived: tried={len(items)} "
                           f"read_errors={sum(ERRS.values())} empty_mask={N_EMPTY} "
                           f"frac_rejected={N_FRAC}")

    df = pd.DataFrame(rows)
    save("N17_siim_ground_truth", df)
    summ = (df.groupby("model")[["gt_iou", "gt_dice", "chance_iou", "pointing_game",
                                 "mass_in_gt", "collapsed"]]
            .mean().round(4).reset_index())
    summ["iou_over_chance"] = (summ.gt_iou / summ.chance_iou).round(3)
    summ["n"] = used
    save("N17b_siim_summary", summ)
    print(summ.to_string(index=False))
    print("  NOTE: localisation QUALITY vs radiologist annotation, from a model")
    print("        fine-tuned IN DOMAIN on held-out SIIM images (answers R3#5).")
    print("  READ THIS: if iou_over_chance < 1.0 the detector is too weak to")
    print("        support a localisation-quality claim -> drop N17/N17b and say so.")

    del SIIM_FP32, SIIM_INT8, _sm, _X, _Y
    gc.collect()
    if CAPS["cuda"]:
        torch.cuda.empty_cache()
    for _d in (FP32, FQ):
        for _k, _m in list(_d.items()):
            _m.to(DEV)
    print("  [ft] paddy models restored to GPU for the figure")


# ------------------- PATCH 2: qualitative CAM panel --------------------------
with stage("FINAL - qualitative CAM panel (patched)"):
    try:
        for a_ in CFG.archs:              # the actual cause of the RuntimeError
            FP32[a_].to(DEV)
            FQ[(a_, "qdq")].to(DEV)
        ps = DRIFT_DF.path.tolist()[:3]
        fig, axes = plt.subplots(len(CFG.archs), 1 + 2 * len(ps),
                                 figsize=(1.6 * (1 + 2 * len(ps)), 1.7 * len(CFG.archs)),
                                 squeeze=False)
        for r, a_ in enumerate(CFG.archs):
            axes[r][0].text(0.5, 0.5, a_, fontsize=7, ha="center", va="center")
            axes[r][0].axis("off")
            for c, p in enumerate(ps):
                x = to_tensor(p)[None].to(DEV)
                with torch.no_grad():
                    t = int(FP32[a_](x).float().argmax(1))
                tt = torch.tensor([t])
                h0 = cam_maps(FP32[a_], CAM_LAYER[a_], x, tt)[0]
                h1 = cam_maps(FQ[(a_, "qdq")], CAM_LAYER[a_], x, tt)[0]
                im = raw_uint8(p)
                for o, h, lab in ((0, h0, "FP32"), (1, h1, "INT8")):
                    ax = axes[r][1 + 2 * c + o]
                    ax.imshow(im)
                    ax.imshow(h, cmap="jet", alpha=.45)
                    ax.axis("off")
                    if r == 0:
                        ax.set_title(lab, fontsize=6)
        fig.suptitle("Grad-CAM: FP32 vs INT8", fontsize=9)
        _fin(fig, "fig_qualitative_cams")
    except Exception as e:
        print("  qualitative panel FAILED:", type(e).__name__, e)
        traceback.print_exc()


# ------------------------------- re-export -----------------------------------
with stage("FINAL - re-export bundle"):
    try:                                   # drop the now-stale skip entries
        SKIPPED[:] = [s for s in SKIPPED
                      if "CELL 15" not in str(s) and "qualitative" not in str(s)]
    except Exception:
        pass

    tex = CFG.out / "tables_latex"
    tex.mkdir(parents=True, exist_ok=True)
    for name in ("N17_siim_ground_truth", "N17b_siim_summary"):
        if name in RESULTS:
            d_ = RESULTS[name]
            try:
                body = d_.head(60).to_latex(index=False, escape=True,
                                            longtable=False, float_format="%.4f")
            except TypeError:
                body = d_.head(60).to_latex(index=False, escape=True)
            (tex / f"{name}.tex").write_text(body)

    mp = CFG.out / "MANIFEST.json"         # patch it, do not re-derive it
    try:
        man = json.loads(mp.read_text())
    except Exception:
        man = {}
    man["tables"] = {k: list(v.shape) for k, v in RESULTS.items()}
    man["siim_cam_layer"] = SIIM_LAYER
    man["rerun_note"] = ("N17/N17b and fig_qualitative_cams were recomputed in a "
                         "follow-up session; every other table and figure is "
                         "carried over unchanged from the full run.")
    mp.write_text(json.dumps(man, indent=2, default=str))

    idx = pd.DataFrame([dict(table=k, rows=v.shape[0], cols=v.shape[1],
                             csv=f"tables/{k}.csv", tex=f"tables_latex/{k}.tex")
                        for k, v in RESULTS.items()])
    idx.to_csv(CFG.out / "TABLE_INDEX.csv", index=False)

    zp = shutil.make_archive("/kaggle/working/quantxai_revision_bundle", "zip",
                             str(CFG.out))
    print(f"BUNDLE: {zp}  ({os.path.getsize(zp) / 1e6:.1f} MB)")
    print(f"TABLES: {len(RESULTS)}   "
          f"FIGURES: {len(list((CFG.out / 'figures').glob('*.png')))}")
    for _n in ("N17_siim_ground_truth", "N17b_siim_summary"):
        print(f"  {'OK      ' if _n in RESULTS else 'MISSING '}{_n}")
    print(f"  {'OK      ' if (CFG.out / 'figures' / 'fig_qualitative_cams.png').exists() else 'MISSING '}"
          f"fig_qualitative_cams.png")
    print(idx.to_string(index=False))
